In [12]:
"""Deep-model fANOVA interaction kernels and curvature for UCI Bike Sharing (hourly).

Stage 1: XGBoost GAM.  Stage 2: the existing OOF depth-2 screen supplies
candidate pairs.  Stage 3: a depth-6 residual model constrained by those pairs.

For each selected pair S={j,k}, Psi_S(x) is a finite fANOVA contrast feature map
over empirical background draws.  K_S = Psi_S Psi_S' / M is therefore PSD and
is induced by the FINAL deep model, not by raw feature distance or the screen.

# ---------------------------------------------------------------------------
# LOGICAL vs. PHYSICAL FEATURES
#
# mnth/hr/weekday are cyclic-encoded (config.add_cyclic_encoding) into
# sin/cos pairs before any model is fit. That split is a PHYSICAL-column
# concern only. Everything else in this script -- the screened-pairs CSV,
# interaction-constraint groupings, the main-effect table's "feature"
# column -- operates on LOGICAL feature names ("mnth", "hr", "weekday",
# "weathersit", ...). config.feature_groups() is the single place that
# expands a logical name back into its physical column(s), and that
# expansion only happens right where a model is actually being built or
# evaluated (interaction_constraints, background_predictions' `changed`
# argument). raw_to_col below therefore maps a logical name to a LIST of
# generic column names, not a single one.
#
# CHANGES IN THIS VERSION
#   - Data comes from ucimlrepo.fetch_ucirepo(id=275), fetched fresh every
#     run; no local file is read.
#   - load_and_prepare() applies C.DROP (removing, among others, the raw
#     date-string column "dteday" -- otherwise XGBoost's dtype handling has
#     no mapping for "object" and raises KeyError: 'object' on .fit()).
#   - mnth/hr/weekday are cyclic-encoded (config.CYCLIC_FEATURES /
#     C.add_cyclic_encoding); weathersit remains a plain category
#     (C.CAT_FEATURES). Every logical feature's physical column(s) -- one
#     for ordinary features, two [sin, cos] for cyclic ones -- are always
#     grouped together in interaction_constraints, so a cyclic feature is
#     treated as one indivisible feature everywhere: the GAM stage, the
#     screened pairs, and the main-effect table.
#   - Target is a plain 1-D numpy array (log1p(cnt) per config.LOG1P_TARGET).
#   - Screened-pairs CSV is validated against the current logical feature
#     set up front, with a specific error pointing at re-running the screen
#     if it's stale, instead of a bare KeyError.
#   - All outputs are written to config.OUT_DIR, your own output folder.
#   - NEW: after computing the main-effect fANOVA table, this script now
#     also plots each main effect as a function of its own feature value
#     (f_j(x) vs x) and saves the plots to config.OUT_DIR -- both a raw
#     (unsmoothed) scatter and a binned-mean curve, per feature. For the
#     3 cyclic features (mnth/hr/weekday), the x-axis is reconstructed
#     back into the original value (e.g. hour 0-23) via the inverse of
#     the sin/cos encoding, rather than plotting in raw sin/cos space.
#
# DEPENDENCY: this script expects a prior "OOF depth-2 screen" step to have
# already written config.OUT_DIR / "bikeshare_hourly_screened_pairs.csv"
# with feature_1, feature_2, screen_gain columns, using LOGICAL feature
# names (see bikeshare_oof_depth2_screen.py).
# ---------------------------------------------------------------------------

# ===========================================================================
# ANNOTATED VERSION -- line-by-line / block-by-block commentary added below.
# No logic has been changed from the version this was generated from.
# ===========================================================================
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBRegressor

# NEW: for the main-effect function plots added at the bottom of this file.
import matplotlib
matplotlib.use("Agg")   # headless/non-interactive backend, safe for scripts
import matplotlib.pyplot as plt

# Make sure the config module (saved in your Downloads folder) is
# importable regardless of where this script itself happens to be run from.
# Python only auto-searches the running script's own directory (plus
# site-packages, etc.) for "import bikeshare_config" -- if you launch this
# script from a different working directory, or if some unrelated module
# happens to shadow the name, the wrong module can get imported silently.
# Explicitly inserting the Downloads folder at the FRONT of sys.path
# (index 0) forces Python to find *this* bikeshare_config.py first.
_DOWNLOADS = Path.home() / "Downloads"
if str(_DOWNLOADS) not in sys.path:
    sys.path.insert(0, str(_DOWNLOADS))

import bikeshare_config as C
# NOTE: the two lines below (a second `import bikeshare_config as C` plus
# `importlib.reload(C)`) are redundant with the import directly above --
# Python has already fully executed bikeshare_config.py by this point, so
# the second `import` is a no-op (it just re-binds C to the same already-
# loaded module object from sys.modules), and reload() re-executes the
# module's top-level code once more, replacing C with a freshly-built
# module object. This is a common leftover from debugging a "my config
# changes aren't showing up" problem (see the module import path/caching
# discussion above) -- reload() forces a live session to pick up on-disk
# changes without restarting the interpreter. It's harmless (just a bit of
# wasted work re-executing bikeshare_config.py's module-level code, e.g.
# creating C.OUT_DIR again via mkdir(exist_ok=True), which is idempotent),
# but isn't needed for a fresh `python this_script.py` process, where the
# plain `import` above already reads the current file from disk exactly
# once. Safe to delete once you've confirmed bikeshare_config.py has the
# functions/attributes you expect.
import importlib
import bikeshare_config as C
importlib.reload(C)


def xgb(depth, constraints=None):
    """Factory for a single XGBoost regressor. Every model fit anywhere in
    this script -- the GAM stage and the deep residual stage -- goes
    through this function, so every model is apples-to-apples except for
    `depth` and `constraints`."""
    # Factory for XGBoost regressors sharing hyperparameters pulled from
    # config.py (SWEEP_ITERS, SWEEP_LR, RANDOM_STATE) so both stages of the
    # pipeline are configured consistently.
    kw = dict(
        n_estimators=C.SWEEP_ITERS,    # boosting rounds
        max_depth=depth,               # 10 for the GAM stage, 6 for the deep residual stage
        learning_rate=C.SWEEP_LR,      # shrinkage
        objective="reg:squarederror",  # standard regression loss
        tree_method="hist",            # histogram-binned splits; required for
                                        # native categorical support below
        n_jobs=-1,                     # use all available CPU cores
        random_state=C.RANDOM_STATE,   # reproducibility
        enable_categorical=True,       # needed for the native-category CAT_FEATURES columns
                                        # (currently just "weathersit") -- lets XGBoost
                                        # split on it as an unordered group of categories
                                        # instead of an ordered numeric scale
    )
    if constraints is not None:
        # interaction_constraints is XGBoost-native: it restricts which
        # features are allowed to appear together along any single root-to-
        # leaf split path. This is how we force "GAM-ness" (each logical
        # feature isolated) or "only these logical pairs may interact" at
        # the model level, rather than post-hoc.
        #
        # Crucially, each element of `constraints` here is a group of
        # PHYSICAL (generic f0/f1/... renamed) column names, already
        # expanded from logical names by the caller via raw_to_col /
        # feature_groups -- e.g. the group for "hr" is two columns
        # (hr's sin and cos), not one, so XGBoost's split-path restriction
        # applies to sin and cos jointly.
        kw["interaction_constraints"] = constraints
    return XGBRegressor(**kw)


def background_predictions(model, Xeval, background, changed, chunk=48):
    """Matrix [point, background]: model(x_changed, b_unchanged)."""
    # This is the core "interventional" evaluator used throughout the
    # fANOVA computation below: for every eval point x (a row of Xeval) and
    # every background row b, compute model(...) with only the `changed`
    # column(s) taken from x -- everything else comes from b. Returns an
    # n x m matrix (n eval points, m background draws), i.e. one row per
    # eval point, one column per background draw.
    #
    # `changed` is a list of physical column names -- for an ordinary
    # feature that's a single column; for a cyclic feature (mnth/hr/
    # weekday) it's [base_sin, base_cos], swapped in together so the pair
    # always represents one consistent underlying value. Swapping sin
    # without also swapping the matching cos (or vice versa) would produce
    # a nonsensical, physically-inconsistent (sin, cos) combination that
    # doesn't correspond to ANY real hour/month/weekday -- hence they must
    # always travel together as one unit, which is exactly what having
    # `changed` be a multi-column list accomplishes.
    n, m = len(Xeval), len(background)
    ans = np.empty((n, m))
    for lo in range(0, n, chunk):
        hi = min(lo + chunk, n)
        # Chunk over eval points so we never materialize an n*m-row frame at
        # once (memory). Tile the background frame once per eval point in
        # this chunk: (hi-lo) copies of the m background rows, stacked
        # vertically. Row block k (of size m) corresponds to eval point k.
        frame = pd.concat([background] * (hi - lo), ignore_index=True)
        for col in changed:
            # Overwrite the "changed" column(s) with the eval point's value,
            # repeated m times per point. Series.repeat (not np.repeat on a
            # bare .to_numpy() array) is essential here: .to_numpy() strips
            # pandas' "category" dtype down to a plain numpy array, and
            # assigning that back into frame[col] would silently replace a
            # categorical column with a numeric one, which would then fail
            # at predict time with an XGBoostError about mismatched dtypes
            # ("the data type doesn't match the one used in the training
            # dataset" -- because the model was trained expecting that
            # column to stay category dtype, but the swapped-in frame now
            # has it as int/float). Series.repeat preserves the original
            # dtype and categories, while producing the identical row
            # ordering: point 0's value repeated m times, then point 1's,
            # ... matching how pd.concat([background]*(hi-lo)) tiles the
            # background rows above.
            frame[col] = Xeval.iloc[lo:hi][col].repeat(m).reset_index(drop=True)
        # Single batched prediction call over the whole chunk-frame (faster
        # than predicting row-by-row), then reshape the flat prediction
        # vector back into (chunk_size, m) -- one row per eval point in
        # this chunk, one column per background draw.
        ans[lo:hi] = model.predict(frame).reshape(hi - lo, m)
    return ans


def model_kernel_curvature(psi, component, k=C.KNN, n_perm=200, seed=0):
    """Support-invariant component curvature on K=Psi Psi'/M's feature graph.

    The roughness R(g) is the count-weighted mean squared graph Laplacian of the
    component. We normalize it by its mean under random permutations of the
    component values over the graph nodes, R(g o pi). The permutation null absorbs
    the graph's size and structure, so Q_S is comparable across components with
    very different numbers of distinct states -- a binary feature on a two-node
    graph no longer inflates. It is also amplitude-invariant, since numerator and
    null scale together. Q_S below 1 is smoother than chance; a degenerate
    few-node graph sits near 1.
    """
    states, inv, counts = np.unique(psi, axis=0, return_inverse=True, return_counts=True)
    values = np.array([component[inv == q].mean() for q in range(len(states))])
    if len(states) < 2 or np.var(component) < 1e-12:
        return 0.0, 0.0, len(states)
    kk = min(k, len(states) - 1)
    d, ix = NearestNeighbors(n_neighbors=kk + 1).fit(states).kneighbors(states)
    d, ix = d[:, 1:], ix[:, 1:]
    fallback = np.median(d[d > 1e-12]) if np.any(d > 1e-12) else 1.0
    scale = np.where(d[:, -1] > 1e-12, d[:, -1], fallback)
    w = np.exp(-d ** 2 / (2 * scale[:, None] ** 2 + 1e-12)); w /= w.sum(axis=1, keepdims=True)

    def roughness(v):
        lap = v - (w * v[ix]).sum(axis=1)
        return float(np.average(lap ** 2, weights=counts))

    raw = roughness(values)
    rng = np.random.default_rng(seed)
    null = float(np.mean([roughness(rng.permutation(values)) for _ in range(n_perm)]))
    return (raw / null if null > 0 else 0.0), raw, len(states)


class SumModel:
    """Full model f = f_1 + g_deep, so first-order contrasts give the model's
    main effects and second-order contrasts give the interactions."""
    def __init__(self, a, b):
        self.a, self.b = a, b

    def predict(self, X):
        return self.a.predict(X) + self.b.predict(X)


def main_effect_table(full, Xeval, background, features, changed_of):
    """Importance A_j and curvature Q_j for each LOGICAL main effect
    (first-order contrast of the full model), computed like the pair
    components. `features` is a list of logical names; changed_of[name] is
    the list of physical column(s) that represent it."""
    # Baseline: predictions on the background set alone (nothing swapped in
    # from eval points), reshaped to a 1 x m row for broadcasting below.
    base = full.predict(background)[None, :]
    rows = []
    for name in features:
        # h: predictions with only this one LOGICAL feature's physical
        # column(s) swapped to the eval point's value(s); everything else
        # stays at background values. For a cyclic feature, both sin and
        # cos are swapped together (changed_of[name] is a 2-element list),
        # so the pair remains a physically consistent point on the circle
        # at every intermediate step of this computation.
        h = background_predictions(full, Xeval, background, changed_of[name])
        psi = h - base                              # first-order fANOVA contrast
        # Average over background draws -> this feature's marginal
        # contribution to the prediction at each eval point.
        comp = psi.mean(axis=1)
        comp = comp - comp.mean()                   # center: A_j is variance
        # Centering makes mean(comp**2) == Var(comp): the classic
        # fANOVA variance-based importance measure.
        # Normalize psi by sqrt(m) before computing curvature so the Psi
        # matrix stays consistent with the K = Psi Psi'/M kernel convention
        # described in the module docstring.
        curv, raw, states = model_kernel_curvature(psi / np.sqrt(psi.shape[1]), comp)
        rows.append({"feature": name, "anova_energy": float(np.mean(comp ** 2)),
                     "component_std": float(np.std(comp)),
                     "model_kernel_curvature": curv, "laplacian_energy": raw,
                     "kernel_states": states})
    return pd.DataFrame(rows).sort_values("anova_energy", ascending=False)


# ---------------------------------------------------------------------------
# NEW: main-effect FUNCTION plots -- f_j(x) vs x, not just the scalar
# importance/curvature summary in main_effect_table().
# ---------------------------------------------------------------------------

def reconstruct_cyclic_value(sin_vals: np.ndarray, cos_vals: np.ndarray, period: float) -> np.ndarray:
    """Invert config.add_cyclic_encoding(): given a feature's sin/cos
    columns (radians = 2*pi*value/period), recover the original value in
    [0, period). Uses atan2 (not plain arcsin/arccos) so the correct
    quadrant -- and therefore the correct point in the cycle -- is
    recovered even though sin alone (or cos alone) is ambiguous about
    which half of the circle you're on."""
    angle = np.arctan2(sin_vals, cos_vals)      # in (-pi, pi]
    angle = np.mod(angle, 2 * np.pi)            # wrap into [0, 2*pi)
    return angle / (2 * np.pi) * period         # rescale to [0, period)


def compute_main_effect_curve(full, Xeval, background, changed_cols):
    """Same first-order fANOVA contrast computation as inside
    main_effect_table(), factored out standalone so it can be called again
    here to get the per-point curve (not just the summary row) without
    modifying main_effect_table() itself."""
    base = full.predict(background)[None, :]
    h = background_predictions(full, Xeval, background, changed_cols)
    psi = h - base
    comp = psi.mean(axis=1)
    comp = comp - comp.mean()
    return comp


def plot_main_effect_functions(full, B, Xte_physical, raw_names, changed_of,
                                groups, background, out_dir, n_bins=30):
    """For every logical main effect, save two plots of the fANOVA
    main-effect FUNCTION f_j(x) against the feature's own value x:
      - "_raw.png": every test-set point plotted as-is (unsmoothed scatter)
      - "_binned.png": the same points binned into n_bins equal-width bins
        along x, with the mean f_j value per bin plotted as a line -- a
        cleaner view of the underlying shape, at the cost of within-bin
        detail.

    x-axis values:
      - Ordinary (non-cyclic) features: the feature's own physical column
        from Xte_physical (pre-generic-rename test set), used directly.
      - Cyclic features (mnth/hr/weekday): reconstructed back into the
        original value (e.g. hour 0-23) via reconstruct_cyclic_value() on
        that feature's sin/cos columns in Xte_physical, rather than
        plotting in raw sin/cos space -- so the x-axis reads "hour of
        day", not two abstract [-1, 1] coordinates.

    y-axis values: the SAME per-point main-effect contribution
    main_effect_table() summarizes into anova_energy/model_kernel_curvature
    for that feature, recomputed here via compute_main_effect_curve() so
    this function can plot the full per-point curve.

    B and Xte_physical must be row-aligned (same row order) -- true here
    because B is built as Xte_physical.copy() with only the column names
    changed, earlier in main().
    """
    print("\nplotting main-effect functions (f_j(x) vs x) for each feature")
    for name in raw_names:
        comp = compute_main_effect_curve(full, B, background, changed_of[name])

        if name in C.CYCLIC_FEATURES:
            period = C.CYCLIC_FEATURES[name]
            sin_col, cos_col = f"{name}_sin", f"{name}_cos"
            x_vals = reconstruct_cyclic_value(
                Xte_physical[sin_col].to_numpy(), Xte_physical[cos_col].to_numpy(), period)
            xlabel = f"{name} (reconstructed from sin/cos, range [0, {period}))"
        else:
            # Ordinary feature: exactly one physical column in its group.
            phys_col = groups[name][0]
            x_vals = Xte_physical[phys_col].to_numpy()
            xlabel = name

        ylabel = f"fANOVA main effect  f_{{{name}}}(x)  (centered)"

        # --- raw (unsmoothed) scatter ---
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(x_vals, comp, s=6, alpha=0.25, color="#4C72B0")
        ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"Main effect (raw): {name}")
        fig.tight_layout()
        raw_path = out_dir / f"bikeshare_main_effect_{name}_raw.png"
        fig.savefig(raw_path, dpi=150)
        plt.close(fig)

        # --- binned mean curve ---
        lo, hi = float(np.min(x_vals)), float(np.max(x_vals))
        bin_edges = np.linspace(lo, hi, n_bins + 1)
        # np.digitize with these interior edges maps values to bins
        # 0..n_bins-1; clip handles the max value landing exactly on the
        # rightmost edge.
        bin_idx = np.clip(np.digitize(x_vals, bin_edges[1:-1]), 0, n_bins - 1)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        bin_means = np.array([
            comp[bin_idx == b].mean() if np.any(bin_idx == b) else np.nan
            for b in range(n_bins)
        ])

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(bin_centers, bin_means, marker="o", color="#DD8452")
        ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel + f"  ({n_bins}-bin mean)")
        ax.set_title(f"Main effect (binned): {name}")
        fig.tight_layout()
        binned_path = out_dir / f"bikeshare_main_effect_{name}_binned.png"
        fig.savefig(binned_path, dpi=150)
        plt.close(fig)

        print(f"  {name}: saved {raw_path.name}, {binned_path.name}")


def load_and_prepare():
    """Fetch the bike-share hourly table via the official ucimlrepo client,
    apply the study's drop list and cyclic encoding, and return
    (X, y, physical_columns)."""
    # Imported here (rather than at module level) so importing this script
    # doesn't require ucimlrepo to be installed unless load_and_prepare()
    # is actually called -- e.g. if you only want xgb()/background_predictions()
    # utilities elsewhere.
    from ucimlrepo import fetch_ucirepo

    print("fetching UCI Bike Sharing (id=275) via ucimlrepo")
    bike_sharing = fetch_ucirepo(id=275)
    X = bike_sharing.data.features.reset_index(drop=True)
    targets = bike_sharing.data.targets.reset_index(drop=True)

    # Remove leakage components (casual/registered directly sum to cnt),
    # identifiers (instant/dteday), and redundant columns (atemp ~
    # correlated with temp, season redundant with mnth) -- see config.py's
    # DROP list. dteday in particular is a raw date STRING (object dtype):
    # if it isn't dropped, XGBoost's dtype handling has no mapping for
    # "object" and raises KeyError: 'object' the moment you try to fit.
    X = X.drop(columns=[c for c in C.DROP if c in X.columns])

    # mnth/hr/weekday -> sin/cos pairs. This is the ONLY place the split
    # happens; everything downstream that needs the logical name goes
    # through C.feature_groups() instead of touching *_sin/*_cos directly.
    # After this call, X's columns for these three are replaced: e.g.
    # "mnth" (one column, values 1-12) becomes "mnth_sin"/"mnth_cos" (two
    # columns, both in [-1, 1]), representing the same information but as
    # a point on a circle rather than a raw integer.
    X = C.add_cyclic_encoding(X)

    # Target: cnt, optionally log1p-transformed (right-skewed count data,
    # so log1p is the usual variance-stabilizing choice), as a plain 1-D
    # numpy array (not a DataFrame column) so downstream positional
    # indexing/arithmetic (train_test_split, ytr - gam.predict(A), etc.)
    # behaves the same way it does everywhere else in this pipeline --
    # avoids a prior bug where a DataFrame target made y[train_idx] try to
    # look up train_idx's integers as column LABELS rather than row
    # positions, raising a KeyError.
    y_raw = targets[C.TARGET].to_numpy(float)
    y = np.log1p(y_raw) if C.LOG1P_TARGET else y_raw

    # Integer-coded categorical that ISN'T cyclic (weathersit) gets pandas
    # "category" dtype so XGBoost's native categorical splits (see
    # enable_categorical=True in xgb()) treat it as an unordered group
    # rather than an ordered numeric quantity.
    for c in C.CAT_FEATURES:
        if c in X.columns:
            X[c] = X[c].astype("category")

    # Safety net: XGBoost (even with enable_categorical=True) only accepts
    # int, float, bool, or category dtypes. Anything else left over here
    # would reproduce the KeyError: 'object' failure from earlier in this
    # pipeline's history -- fail loudly and specifically instead, naming
    # the offending column(s), rather than letting it resurface as a
    # cryptic error three layers down inside XGBoost.
    bad = [c for c in X.columns if X[c].dtype == object]
    if bad:
        raise TypeError(
            f"Non-numeric, non-categorical columns remain after preprocessing: {bad}. "
            "Add them to config.DROP, config.CAT_FEATURES, or config.CYCLIC_FEATURES."
        )

    return X, y, list(X.columns)


def main():
    X, y, phys_names = load_and_prepare()
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=C.TEST_SIZE,
                                           random_state=C.RANDOM_STATE)
    # Reset indices so later .iloc-based positional access (in
    # background_predictions) lines up cleanly.
    Xtr, Xte = Xtr.reset_index(drop=True), Xte.reset_index(drop=True)
    # Rename PHYSICAL columns to generic f0, f1, ... -- keeps XGBoost's
    # interaction_constraints syntax simple/stable, independent of the
    # original column names. Category dtype survives the rename/copy
    # (renaming columns doesn't touch dtype).
    cols = [f"f{i}" for i in range(X.shape[1])]
    phys_to_generic = dict(zip(phys_names, cols))
    A, B = Xtr.copy(), Xte.copy(); A.columns = cols; B.columns = cols

    # groups: logical name -> physical column(s), e.g.
    # {"mnth": ["mnth_sin", "mnth_cos"], "weathersit": ["weathersit"], ...}
    # (see config.feature_groups() -- it inspects which *_sin/*_cos pairs
    # exist and reconstructs the logical grouping from that).
    #
    # raw_to_col: logical name -> GENERIC column(s) -- the SAME grouping,
    # but with each physical column name translated through
    # phys_to_generic into its f0/f1/... form. This is the dict everything
    # below actually consumes (interaction_constraints,
    # background_predictions's `changed` argument), since A/B/background
    # are all in generic-column form, not original bike-share column names.
    groups = C.feature_groups(phys_names)
    raw_to_col = {name: [phys_to_generic[p] for p in phys] for name, phys in groups.items()}
    raw_names = list(groups.keys())   # logical feature names, e.g. "mnth", "hr", ...

    # Load the top-8 candidate interaction pairs from the prior screening
    # step (see bikeshare_oof_depth2_screen.py), which screens and writes
    # by LOGICAL feature name -- so feature_1/feature_2 in the CSV are
    # e.g. "mnth"/"hr", never "mnth_sin"/"hr_cos".
    screened_path = C.OUT_DIR / "bikeshare_hourly_screened_pairs.csv"
    if not screened_path.exists():
        # Fail with a specific, actionable message rather than letting
        # pd.read_csv raise its own generic "file not found" error.
        raise FileNotFoundError(
            f"Expected screened-pairs file not found: {screened_path}\n"
            "Run the depth-2 OOF interaction screen on the bike-share data "
            "first and save its output (feature_1, feature_2, screen_gain "
            "columns) to this path before running this script."
        )
    # .head(8): only the top 8 candidate pairs (by whatever order the
    # screen script sorted the CSV in -- descending screen_gain) become
    # actual interaction terms in the deep model below.
    screened = pd.read_csv(screened_path).head(8)
    # Guard against a stale screened-pairs CSV: if load_and_prepare()'s
    # logical feature set has changed since the screen was last run (e.g.
    # a feature was added to DROP, or the cyclic-encoding set changed),
    # raw_to_col won't have an entry for some name, and a bare dict lookup
    # in the list comprehension below would fail with an unhelpful
    # KeyError buried inside a comprehension. Check every name referenced
    # by the CSV up front, collect ALL the unknown ones (not just the
    # first), and point directly at the fix.
    unknown = sorted({
        name for row in screened.itertuples(index=False)
        for name in (row.feature_1, row.feature_2)
        if name not in raw_to_col
    })
    if unknown:
        raise KeyError(
            f"{screened_path.name} references feature(s) not in the current "
            f"logical feature set: {unknown}. This usually means the "
            "screened-pairs CSV is stale relative to load_and_prepare()/"
            "config.feature_groups(). Re-run bikeshare_oof_depth2_screen.py "
            "to regenerate it, then re-run this script."
        )
    # Each pair entry is now (list_of_generic_cols_for_feature_1,
    # list_of_generic_cols_for_feature_2) -- a singleton list for an
    # ordinary feature (e.g. ["f7"] for temp), a two-element [sin_col,
    # cos_col] list for a cyclic one (e.g. ["f2", "f3"] for hr).
    pairs = [(raw_to_col[row.feature_1], raw_to_col[row.feature_2])
             for row in screened.itertuples(index=False)]
    target_desc = f"log1p({C.TARGET})" if C.LOG1P_TARGET else C.TARGET
    print(f"fit final deep residual model; target={target_desc}")
    # Stage 1 (GAM): depth-10 XGBoost, but interaction_constraints groups
    # each LOGICAL feature's generic column(s) together and forbids any
    # cross-feature split path -- despite depth 10, it cannot build
    # cross-feature interactions. A cyclic feature's sin/cos pair may
    # combine with each other within a tree (they jointly represent one
    # value -- e.g. a tree needs both sin(hr) and cos(hr) together to
    # approximate an arbitrary smooth function of hour-of-day) but not
    # with anything else. Result is effectively additive per logical
    # feature, with deep single-feature (or single-cyclic-feature)
    # nonlinearity allowed.
    gam = xgb(10, [raw_to_col[name] for name in raw_names]).fit(A, ytr)
    # Stage 2 (deep residual): depth-6 XGBoost fit on the GAM's residuals
    # (ytr - gam.predict(A)), constrained so only the 8 screened LOGICAL
    # pairs may interact -- each pair's constraint group is the UNION
    # (list concatenation, `a + b`) of both features' generic column(s).
    # E.g. a pair (hr, weathersit) becomes one constraint group
    # ["f2","f3","f9"] (hr's sin, hr's cos, weathersit), so trees built for
    # that pair may split on any combination of those three columns. This
    # is "the depth-6 residual model constrained by those pairs" from the
    # module docstring.
    deep = xgb(6, [a + b for a, b in pairs]).fit(A, ytr - gam.predict(A))
    pred = gam.predict(B) + deep.predict(B)
    print(f"  held-out {target_desc} MAE: GAM+screened-deep={np.abs(pred-yte).mean():.4f}")

    # Fixed-seed sample of 96 background rows from the training set, used
    # as the reference distribution for all fANOVA interventions below.
    rng = np.random.default_rng(C.RANDOM_STATE)
    background = A.iloc[rng.choice(len(A), size=96, replace=False)].reset_index(drop=True)
    # Deep model's baseline predictions on the background set alone (no
    # eval-point values swapped in).
    base = deep.predict(background)[None, :]
    records = []
    print("construct deep fANOVA PSD kernels and component curvature")
    for (a, b), row in zip(pairs, screened.itertuples(index=False)):
        # Classical fANOVA inclusion-exclusion for a second-order (pairwise)
        # interaction between LOGICAL features j and k, evaluated on the
        # test set B against the background sample, using the deep
        # residual model only:
        #
        #   f_{jk}(x) ~= E_b[model(x_j, x_k, b_rest)]
        #              - E_b[model(x_j, b_rest)]
        #              - E_b[model(x_k, b_rest)]
        #              + E_b[model(b)]
        #
        # a and b are each lists of generic column(s) -- a + b swaps both
        # features' full physical representation in at once (e.g. both
        # hr_sin and hr_cos together, if hr is one of the pair), so "x_j,
        # x_k" above always means the physically-consistent full
        # representation of each logical feature, never a half-swapped
        # sin-without-cos state.
        hab = background_predictions(deep, B, background, a + b)  # both features swapped
        ha = background_predictions(deep, B, background, a)       # only a swapped
        hb = background_predictions(deep, B, background, b)       # only b swapped
        # Inclusion-exclusion combination: cancels both main effects and the
        # baseline, isolating the pure pairwise interaction contrast. This
        # psi matrix (per-eval-point, per-background-draw) is exactly what
        # forms K_S = Psi_S Psi_S' / M -- the PSD kernel induced by the
        # deep model's actual interaction structure, per the module
        # docstring.
        psi = hab - ha - hb + base                 # finite fANOVA feature map
        component = psi.mean(axis=1)               # f_{jk} under this measure
        component = component - component.mean()    # center: A_S is variance, not 2nd moment
        # Curvature diagnostic for this interaction: how smooth/coherent
        # the interaction component is over its own model-induced graph.
        curvature, raw, states = model_kernel_curvature(psi / np.sqrt(psi.shape[1]), component)
        records.append({"feature_1": row.feature_1, "feature_2": row.feature_2,
                        "screen_gain": row.screen_gain,
                        "anova_energy": float(np.mean(component ** 2)),
                        "component_std": float(np.std(component)),
                        "model_kernel_curvature": curvature,
                        "laplacian_energy": raw, "kernel_states": states,
                        # Rank of the raw psi matrix: a low-rank psi
                        # suggests this interaction's kernel has few
                        # effective degrees of freedom (simple/degenerate),
                        # rather than being richly structured.
                        "kernel_rank": int(np.linalg.matrix_rank(psi))})
    out = pd.DataFrame(records).sort_values("anova_energy", ascending=False)
    print("\nDeep-model pair fANOVA importance + model-kernel curvature:")
    print(out.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    out.to_csv(C.OUT_DIR / "bikeshare_hourly_deep_anova_kernel_curvature.csv", index=False)
    np.savez(C.OUT_DIR / "bikeshare_hourly_deep_anova_kernel_curvature.npz",
             **{col: out[col].to_numpy() for col in out.columns})
    print(f"\nsaved -> {C.OUT_DIR / 'bikeshare_hourly_deep_anova_kernel_curvature.csv'}")

    # Wrap GAM + deep residual as a single combined model so main effects
    # are computed against the FULL model, not just the residual stage --
    # a main effect belongs to the whole prediction, not just whatever the
    # residual stage happened to pick up.
    full = SumModel(gam, deep)
    changed_of = {name: raw_to_col[name] for name in raw_names}
    mains = main_effect_table(full, B, background, raw_names, changed_of)
    print("\nMain-effect fANOVA importance + model-kernel curvature:")
    print(mains.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    mains.to_csv(C.OUT_DIR / "bikeshare_hourly_main_effect_curvature.csv", index=False)

    # NEW: plot each main effect as a function of its own feature value
    # (both a raw scatter and a binned-mean curve per feature), saved to
    # C.OUT_DIR. Xte (not B) is passed for the x-axis values because Xte
    # still has the original physical column names (mnth_sin, hr_cos,
    # weathersit, ...) needed for the cyclic-value reconstruction and for
    # readable non-cyclic feature names -- B has already been renamed to
    # generic f0/f1/... columns for modeling.
    plot_main_effect_functions(full, B, Xte, raw_names, changed_of, groups,
                                background, C.OUT_DIR)


if __name__ == "__main__":
    main()

fetching UCI Bike Sharing (id=275) via ucimlrepo
fit final deep residual model; target=log1p(cnt)
  held-out log1p(cnt) MAE: GAM+screened-deep=0.2434
construct deep fANOVA PSD kernels and component curvature

Deep-model pair fANOVA importance + model-kernel curvature:
 feature_1  feature_2  screen_gain  anova_energy  component_std  model_kernel_curvature  laplacian_energy  kernel_states  kernel_rank
        hr    weekday      0.03811       0.09176        0.30291                 0.10504           0.01000            168           96
        hr workingday      0.05154       0.02413        0.15533                 0.24019           0.00801             48           48
      temp        hum      0.00124       0.00441        0.06644                 0.08136           0.00056            748           96
weathersit        hum      0.00136       0.00279        0.05280                 0.04258           0.00019            190           96
        hr        hum      0.00278       0.00201        0.044

In [24]:
"""OOF depth-2 interaction screen for UCI Bike Sharing (hourly).

For every candidate LOGICAL feature pair (j, k), cross-fits two depth-
C.REF_DEPTH XGBoost models under C.SWEEP_CV-fold out-of-fold (OOF)
evaluation:
  1. A fully additive baseline where no logical features are allowed to
     interact (interaction_constraints = one logical feature per group).
  2. The same shape, except j and k are additionally allowed to interact.

screen_gain = baseline_OOF_MAE - pair_OOF_MAE: positive means letting j and
k interact measurably improves held-out predictions beyond what each
contributes on its own. Ranking by this gives the candidate pairs that the
downstream fANOVA kernel-curvature script consumes.

Writes the ranked pairs to config.OUT_DIR / "bikeshare_hourly_screened_pairs.csv"
with feature_1, feature_2, screen_gain columns -- exactly the file the
fANOVA script expects.

# ---------------------------------------------------------------------------
# LOGICAL vs. PHYSICAL FEATURES
#
# mnth/hr/weekday are cyclic-encoded (config.add_cyclic_encoding) into
# sin/cos pairs before any model is fit -- that's a PHYSICAL-column-level
# concern. But screen_gain, and everything written to the CSV, operates on
# LOGICAL feature names ("mnth", "hr", "weekday", "weathersit", ...): the
# sin/cos split is collapsed back into one name via config.feature_groups(),
# and interaction_constraints groups a logical feature's physical column(s)
# together so e.g. mnth_sin and mnth_cos are always treated as a single
# indivisible unit, never screened as if they were two separate features.
# This keeps the CSV -- and anything else that reads it -- talking about
# "mnth" and "hr", not "mnth_sin"/"mnth_cos"/"hr_sin"/"hr_cos".
# ---------------------------------------------------------------------------
"""
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

# Make sure the config module (saved in your Downloads folder) is
# importable regardless of where this script itself happens to be run from.
_DOWNLOADS = Path.home() / "Downloads"
if str(_DOWNLOADS) not in sys.path:
    sys.path.insert(0, str(_DOWNLOADS))

import importlib
import bikeshare_config as C
importlib.reload(C)
import bikeshare_config as C


def xgb(depth, constraints=None):
    # Same factory shape as the fANOVA script, but using the *reference*
    # hyperparameters (C.REF_ITERS / C.REF_LR / C.REF_DEPTH) that define
    # this study's screening/geometry model.
    kw = dict(n_estimators=C.REF_ITERS, max_depth=depth, learning_rate=C.REF_LR,
              objective="reg:squarederror", tree_method="hist", n_jobs=-1,
              random_state=C.RANDOM_STATE, enable_categorical=True)
    if constraints is not None:
        kw["interaction_constraints"] = constraints
    return XGBRegressor(**kw)


def load_and_prepare():
    """Fetch the bike-share hourly table via the official ucimlrepo client,
    apply the study's drop list and cyclic encoding, and return
    (X, y, physical_columns). Mirrors the fANOVA script's
    load_and_prepare() so both scripts agree on the exact feature set."""
    from ucimlrepo import fetch_ucirepo

    print("fetching UCI Bike Sharing (id=275) via ucimlrepo")
    bike_sharing = fetch_ucirepo(id=275)
    X = bike_sharing.data.features.reset_index(drop=True)
    targets = bike_sharing.data.targets.reset_index(drop=True)

    # Leakage/identifier/redundant columns -- see config.DROP. Removes the
    # object-dtype "dteday" column among others.
    X = X.drop(columns=[c for c in C.DROP if c in X.columns])

    # mnth/hr/weekday -> sin/cos pairs (see config.CYCLIC_FEATURES);
    # weathersit is left for the category-dtype step below. This is the
    # ONLY place the sin/cos split happens -- everything downstream that
    # needs the *logical* name goes through C.feature_groups() instead.
    X = C.add_cyclic_encoding(X)

    # Target: cnt, optionally log1p-transformed, as a plain 1-D numpy array
    # so y[train_idx] / y[test_idx] below are ordinary positional indexing.
    y_raw = targets[C.TARGET].to_numpy(float)
    y = np.log1p(y_raw) if C.LOG1P_TARGET else y_raw

    for c in C.CAT_FEATURES:
        if c in X.columns:
            X[c] = X[c].astype("category")

    bad = [c for c in X.columns if X[c].dtype == object]
    if bad:
        raise TypeError(
            f"Non-numeric, non-categorical columns remain after preprocessing: {bad}. "
            "Add them to config.DROP, config.CAT_FEATURES, or config.CYCLIC_FEATURES."
        )

    return X.reset_index(drop=True), y, list(X.columns)


def oof_predictions(constraints, X, y, n_splits, seed):
    """Cross-fitted (out-of-fold) predictions for one interaction-constraint
    configuration at depth=C.REF_DEPTH. Every row is predicted only by a
    model that never trained on it -- this is what makes screen_gain a fair
    held-out comparison rather than an in-sample (overfit-prone) gain."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    pred = np.empty(len(X))
    for train_idx, test_idx in kf.split(X):
        model = xgb(C.REF_DEPTH, constraints)
        model.fit(X.iloc[train_idx], y[train_idx])
        pred[test_idx] = model.predict(X.iloc[test_idx])
    return pred


def main():
    X, y, phys_cols = load_and_prepare()

    # groups: logical name -> list of physical column(s), e.g.
    # {"mnth": ["mnth_sin", "mnth_cos"], "weathersit": ["weathersit"], ...}
    # logical_names is what gets screened and written to the CSV.
    groups = C.feature_groups(phys_cols)
    logical_names = list(groups.keys())

    n_splits = C.SWEEP_CV or 5
    print(f"n={len(X)} rows, {len(logical_names)} logical features "
          f"({len(phys_cols)} physical columns), {n_splits}-fold OOF, "
          f"depth={C.REF_DEPTH}")

    # Baseline: every LOGICAL feature isolated -- a cyclic feature's sin/cos
    # pair is grouped together (so they may combine with each other) but
    # forbidden from interacting with anything else. Same constraint shape
    # used for the GAM stage of the downstream fANOVA script.
    print("fitting additive baseline (all logical features isolated)")
    baseline_constraints = list(groups.values())
    baseline_pred = oof_predictions(baseline_constraints, X, y, n_splits, C.RANDOM_STATE)
    baseline_mae = float(np.abs(baseline_pred - y).mean())
    print(f"  baseline OOF MAE: {baseline_mae:.5f}")

    records = []
    pairs = list(combinations(logical_names, 2))
    print(f"screening {len(pairs)} candidate logical pairs "
          f"({len(pairs) + 1} model families x {n_splits} folds total)")
    for i, (j, k) in enumerate(pairs, 1):
        # Same additive shape as the baseline, except j and k's physical
        # column(s) are combined into ONE interacting group.
        constraints = ([grp for name, grp in groups.items() if name not in (j, k)]
                        + [groups[j] + groups[k]])
        pair_pred = oof_predictions(constraints, X, y, n_splits, C.RANDOM_STATE)
        pair_mae = float(np.abs(pair_pred - y).mean())
        gain = baseline_mae - pair_mae   # positive = interaction helps OOF error
        records.append({"feature_1": j, "feature_2": k,
                         "screen_gain": gain,
                         "pair_oof_mae": pair_mae,
                         "baseline_oof_mae": baseline_mae})
        if i % 5 == 0 or i == len(pairs):
            print(f"  [{i}/{len(pairs)}] {j} x {k}: gain={gain:.5f}")

    out = (pd.DataFrame(records)
           .sort_values("screen_gain", ascending=False)
           .reset_index(drop=True))
    print("\nTop candidate interaction pairs by OOF screen gain:")
    print(out.head(15).to_string(index=False, float_format=lambda v: f"{v:.5f}"))

    out_path = C.OUT_DIR / "bikeshare_hourly_screened_pairs.csv"
    out.to_csv(out_path, index=False)
    print(f"\nsaved -> {out_path}")


if __name__ == "__main__":
    main()

fetching UCI Bike Sharing (id=275) via ucimlrepo
n=17379 rows, 10 logical features (13 physical columns), 5-fold OOF, depth=2
fitting additive baseline (all logical features isolated)
  baseline OOF MAE: 0.44767
screening 45 candidate logical pairs (46 model families x 5 folds total)
  [5/45] mnth x workingday: gain=-0.00007
  [10/45] hr x weekday: gain=0.03811
  [15/45] hr x temp: gain=0.00028
  [20/45] weekday x workingday: gain=-0.00060
  [25/45] yr x holiday: gain=-0.00169
  [30/45] yr x windspeed: gain=-0.00290
  [35/45] holiday x windspeed: gain=-0.00081
  [40/45] weathersit x temp: gain=-0.00133
  [45/45] hum x windspeed: gain=0.00070

Top candidate interaction pairs by OOF screen gain:
 feature_1  feature_2  screen_gain  pair_oof_mae  baseline_oof_mae
        hr workingday      0.05154       0.39613           0.44767
        hr    weekday      0.03811       0.40956           0.44767
        hr    holiday      0.00400       0.44367           0.44767
        hr        hum      0.

In [2]:
"""OOF depth-2 interaction screen for the California Housing dataset.

For every candidate feature pair (j, k), cross-fits two depth-C.REF_DEPTH
XGBoost models under C.SWEEP_CV-fold out-of-fold (OOF) evaluation:
  1. A fully additive baseline where no features are allowed to interact
     (interaction_constraints = one feature per group).
  2. The same shape, except j and k are additionally allowed to interact.

screen_gain = baseline_OOF_MAE - pair_OOF_MAE: positive means letting j and
k interact measurably improves held-out predictions beyond what each
contributes on its own. Ranking by this gives the candidate pairs that the
downstream fANOVA kernel-curvature script consumes.

Writes the ranked pairs to config.OUT_DIR / "california_housing_screened_pairs.csv"
with feature_1, feature_2, screen_gain columns.

# ---------------------------------------------------------------------------
# NAMING / FILE SEPARATION
# California Housing counterpart to bikeshare_oof_depth2_screen.py. Uses
# its own config module (california_housing_config.py) and its own
# C.OUT_DIR (Downloads/california_housing_artifacts), and writes its own
# uniquely-named CSV. Nothing here touches bikeshare_config.py, the
# bikeshare_artifacts folder, or any bikeshare_hourly_* file.
#
# NO CYCLIC ENCODING
# Unlike the bike-share study, this dataset has no cyclic (hour/month/
# weekday-style) features and no categorical features, so there is no
# cyclic-encoding or category-casting step here, and every feature is its
# own single-column group -- no config.feature_groups()-style logical/
# physical translation is needed. `cols` below IS the feature list used
# for screening, directly.
# ---------------------------------------------------------------------------
"""
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

# Make sure the config module (saved in your Downloads folder) is
# importable regardless of where this script itself happens to be run from.
_DOWNLOADS = Path.home() / "Downloads"
if str(_DOWNLOADS) not in sys.path:
    sys.path.insert(0, str(_DOWNLOADS))

import california_housing_config as C


def xgb(depth, constraints=None):
    # Same factory shape as the fANOVA script, using the *reference*
    # hyperparameters (C.REF_ITERS / C.REF_LR / C.REF_DEPTH) that define
    # this study's screening/geometry model.
    kw = dict(n_estimators=C.REF_ITERS, max_depth=depth, learning_rate=C.REF_LR,
              objective="reg:squarederror", tree_method="hist", n_jobs=-1,
              random_state=C.RANDOM_STATE)
    if constraints is not None:
        kw["interaction_constraints"] = constraints
    return XGBRegressor(**kw)


def load_and_prepare():
    """Fetch California Housing (via sklearn) and return (X, y, cols)."""
    print("loading California Housing data (sklearn.datasets.fetch_california_housing)")
    X, y_raw = C.load_california_housing()

    # No-op today (C.DROP is empty), kept for structural symmetry with the
    # bike-share script in case a column ever needs excluding later.
    X = X.drop(columns=[c for c in C.DROP if c in X.columns])

    y = np.log1p(y_raw) if C.LOG1P_TARGET else y_raw

    # No-op today (C.CAT_FEATURES is empty): every column here is already
    # a continuous numeric quantity.
    for c in C.CAT_FEATURES:
        if c in X.columns:
            X[c] = X[c].astype("category")

    bad = [c for c in X.columns if X[c].dtype == object]
    if bad:
        raise TypeError(
            f"Non-numeric, non-categorical columns remain after preprocessing: {bad}. "
            "Add them to config.DROP or config.CAT_FEATURES."
        )

    return X.reset_index(drop=True), y, list(X.columns)


def oof_predictions(constraints, X, y, n_splits, seed):
    """Cross-fitted (out-of-fold) predictions for one interaction-constraint
    configuration at depth=C.REF_DEPTH. Every row is predicted only by a
    model that never trained on it -- this is what makes screen_gain a fair
    held-out comparison rather than an in-sample (overfit-prone) gain."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    pred = np.empty(len(X))
    for train_idx, test_idx in kf.split(X):
        model = xgb(C.REF_DEPTH, constraints)
        model.fit(X.iloc[train_idx], y[train_idx])
        pred[test_idx] = model.predict(X.iloc[test_idx])
    return pred


def main():
    X, y, cols = load_and_prepare()
    n_splits = C.SWEEP_CV or 5
    print(f"n={len(X)} rows, {len(cols)} candidate features, "
          f"{n_splits}-fold OOF, depth={C.REF_DEPTH}")

    # Baseline: every feature isolated (purely additive, nothing allowed to
    # interact) -- the same constraint shape used for the GAM stage in the
    # downstream fANOVA script.
    print("fitting additive baseline (all features isolated)")
    baseline_constraints = [[c] for c in cols]
    baseline_pred = oof_predictions(baseline_constraints, X, y, n_splits, C.RANDOM_STATE)
    baseline_mae = float(np.abs(baseline_pred - y).mean())
    print(f"  baseline OOF MAE: {baseline_mae:.5f}")

    records = []
    pairs = list(combinations(cols, 2))
    print(f"screening {len(pairs)} candidate pairs "
          f"({len(pairs) + 1} model families x {n_splits} folds total)")
    for i, (j, k) in enumerate(pairs, 1):
        constraints = [[c] for c in cols if c not in (j, k)] + [[j, k]]
        pair_pred = oof_predictions(constraints, X, y, n_splits, C.RANDOM_STATE)
        pair_mae = float(np.abs(pair_pred - y).mean())
        gain = baseline_mae - pair_mae   # positive = interaction helps OOF error
        records.append({"feature_1": j, "feature_2": k,
                         "screen_gain": gain,
                         "pair_oof_mae": pair_mae,
                         "baseline_oof_mae": baseline_mae})
        if i % 5 == 0 or i == len(pairs):
            print(f"  [{i}/{len(pairs)}] {j} x {k}: gain={gain:.5f}")

    out = (pd.DataFrame(records)
           .sort_values("screen_gain", ascending=False)
           .reset_index(drop=True))
    print("\nTop candidate interaction pairs by OOF screen gain:")
    print(out.head(15).to_string(index=False, float_format=lambda v: f"{v:.5f}"))

    out_path = C.OUT_DIR / "california_housing_screened_pairs.csv"
    out.to_csv(out_path, index=False)
    print(f"\nsaved -> {out_path}")


if __name__ == "__main__":
    main()

loading California Housing data (sklearn.datasets.fetch_california_housing)
n=20640 rows, 8 candidate features, 5-fold OOF, depth=2
fitting additive baseline (all features isolated)
  baseline OOF MAE: 0.12926
screening 28 candidate pairs (29 model families x 5 folds total)
  [5/28] MedInc x AveOccup: gain=0.00070
  [10/28] HouseAge x Population: gain=0.00011
  [15/28] AveRooms x Population: gain=0.00015
  [20/28] AveBedrms x AveOccup: gain=0.00015
  [25/28] Population x Longitude: gain=0.00024
  [28/28] Latitude x Longitude: gain=0.01309

Top candidate interaction pairs by OOF screen gain:
 feature_1 feature_2  screen_gain  pair_oof_mae  baseline_oof_mae
  Latitude Longitude      0.01309       0.11617           0.12926
  AveOccup Longitude      0.00169       0.12757           0.12926
  HouseAge  AveOccup      0.00141       0.12785           0.12926
  AveOccup  Latitude      0.00093       0.12832           0.12926
    MedInc Longitude      0.00083       0.12843           0.12926
    Me

In [13]:
"""Deep-model fANOVA interaction kernels and curvature for California Housing.

Stage 1: XGBoost GAM.  Stage 2: the existing OOF depth-2 screen supplies
candidate pairs.  Stage 3: a depth-6 residual model constrained by those pairs.

For each selected pair S={j,k}, Psi_S(x) is a finite fANOVA contrast feature map
over empirical background draws.  K_S = Psi_S Psi_S' / M is therefore PSD and
is induced by the FINAL deep model, not by raw feature distance or the screen.

# ---------------------------------------------------------------------------
# NAMING / FILE SEPARATION
# California Housing counterpart to bikeshare_deep_anova_kernel_curvature.py.
# Own config module (california_housing_config.py), own C.OUT_DIR
# (Downloads/california_housing_artifacts), own uniquely-named outputs
# (california_housing_*). Nothing here touches bikeshare_config.py, the
# bikeshare_artifacts folder, or any bikeshare_hourly_* file.
#
# NO CYCLIC ENCODING
# This dataset has no cyclic (hour/month/weekday-style) or categorical
# features, so unlike the bike-share version there is no
# config.feature_groups()-style logical/physical translation here --
# raw_to_col maps each feature name directly to a single generic column,
# and `changed` arguments to background_predictions are always singleton
# lists. Correspondingly, the main-effect PLOTS added below have no
# cyclic-reconstruction branch either -- every feature's x-axis is just
# its own raw value, used directly.
#
# DEPENDENCY: this script expects a prior "OOF depth-2 screen" step to have
# already written config.OUT_DIR / "california_housing_screened_pairs.csv"
# with feature_1, feature_2, screen_gain columns (see
# california_housing_oof_depth2_screen.py).
#
# NEW: after computing the main-effect fANOVA table, this script now also
# plots each main effect as a function of its own feature value (f_j(x)
# vs x) and saves the plots to config.OUT_DIR -- both a raw (unsmoothed)
# scatter and a binned-mean curve, per feature (same approach as the
# bike-share version's main-effect plots, minus the cyclic-reconstruction
# step since nothing here needs it).
# ---------------------------------------------------------------------------
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBRegressor

# NEW: for the main-effect function plots added at the bottom of this file.
import matplotlib
matplotlib.use("Agg")   # headless/non-interactive backend, safe for scripts
import matplotlib.pyplot as plt

# Make sure the config module (saved in your Downloads folder) is
# importable regardless of where this script itself happens to be run from.
_DOWNLOADS = Path.home() / "Downloads"
if str(_DOWNLOADS) not in sys.path:
    sys.path.insert(0, str(_DOWNLOADS))

import california_housing_config as C


def xgb(depth, constraints=None):
    # Factory for XGBoost regressors sharing hyperparameters pulled from
    # config.py (SWEEP_ITERS, SWEEP_LR, RANDOM_STATE) so both stages of the
    # pipeline are configured consistently.
    kw = dict(n_estimators=C.SWEEP_ITERS, max_depth=depth, learning_rate=C.SWEEP_LR,
              objective="reg:squarederror", tree_method="hist", n_jobs=-1,
              random_state=C.RANDOM_STATE)
    if constraints is not None:
        # interaction_constraints is XGBoost-native: it restricts which
        # features are allowed to appear together along any single root-to-
        # leaf split path. This is how we force "GAM-ness" (each feature
        # isolated) or "only these pairs may interact" at the model level,
        # rather than post-hoc.
        kw["interaction_constraints"] = constraints
    return XGBRegressor(**kw)


def background_predictions(model, Xeval, background, changed, chunk=48):
    """Matrix [point, background]: model(x_changed, b_unchanged)."""
    # For every eval point x and every background row b, compute model(...)
    # with only the `changed` column(s) taken from x -- everything else
    # comes from b. Returns an n x m matrix (n eval points, m background
    # draws).
    n, m = len(Xeval), len(background)
    ans = np.empty((n, m))
    for lo in range(0, n, chunk):
        hi = min(lo + chunk, n)
        # Chunk over eval points so we never materialize an n*m-row frame at
        # once (memory). Tile the background frame once per eval point in
        # this chunk: (hi-lo) copies of the m background rows.
        frame = pd.concat([background] * (hi - lo), ignore_index=True)
        for col in changed:
            # Series.repeat (not np.repeat on a bare .to_numpy() array)
            # preserves dtype -- matters if this dataset ever gains a
            # categorical column, and costs nothing when it doesn't.
            frame[col] = Xeval.iloc[lo:hi][col].repeat(m).reset_index(drop=True)
        # Single batched prediction call over the whole chunk-frame, then
        # reshape back into (chunk_size, m).
        ans[lo:hi] = model.predict(frame).reshape(hi - lo, m)
    return ans


def model_kernel_curvature(psi, component, k=C.KNN, n_perm=200, seed=0):
    """Support-invariant component curvature on K=Psi Psi'/M's feature graph.

    The roughness R(g) is the count-weighted mean squared graph Laplacian of the
    component. We normalize it by its mean under random permutations of the
    component values over the graph nodes, R(g o pi). The permutation null absorbs
    the graph's size and structure, so Q_S is comparable across components with
    very different numbers of distinct states -- a binary feature on a two-node
    graph no longer inflates. It is also amplitude-invariant, since numerator and
    null scale together. Q_S below 1 is smoother than chance; a degenerate
    few-node graph sits near 1.
    """
    states, inv, counts = np.unique(psi, axis=0, return_inverse=True, return_counts=True)
    values = np.array([component[inv == q].mean() for q in range(len(states))])
    if len(states) < 2 or np.var(component) < 1e-12:
        return 0.0, 0.0, len(states)
    kk = min(k, len(states) - 1)
    d, ix = NearestNeighbors(n_neighbors=kk + 1).fit(states).kneighbors(states)
    d, ix = d[:, 1:], ix[:, 1:]
    fallback = np.median(d[d > 1e-12]) if np.any(d > 1e-12) else 1.0
    scale = np.where(d[:, -1] > 1e-12, d[:, -1], fallback)
    w = np.exp(-d ** 2 / (2 * scale[:, None] ** 2 + 1e-12)); w /= w.sum(axis=1, keepdims=True)

    def roughness(v):
        lap = v - (w * v[ix]).sum(axis=1)
        return float(np.average(lap ** 2, weights=counts))

    raw = roughness(values)
    rng = np.random.default_rng(seed)
    null = float(np.mean([roughness(rng.permutation(values)) for _ in range(n_perm)]))
    return (raw / null if null > 0 else 0.0), raw, len(states)


class SumModel:
    """Full model f = f_1 + g_deep, so first-order contrasts give the model's
    main effects and second-order contrasts give the interactions."""
    def __init__(self, a, b):
        self.a, self.b = a, b

    def predict(self, X):
        return self.a.predict(X) + self.b.predict(X)


def main_effect_table(full, Xeval, background, features, changed_of):
    """Importance A_j and curvature Q_j for each main effect (first-order
    contrast of the full model), computed like the pair components."""
    base = full.predict(background)[None, :]
    rows = []
    for name in features:
        h = background_predictions(full, Xeval, background, changed_of[name])
        psi = h - base                              # first-order fANOVA contrast
        comp = psi.mean(axis=1)
        comp = comp - comp.mean()                   # center: A_j is variance
        curv, raw, states = model_kernel_curvature(psi / np.sqrt(psi.shape[1]), comp)
        rows.append({"feature": name, "anova_energy": float(np.mean(comp ** 2)),
                     "component_std": float(np.std(comp)),
                     "model_kernel_curvature": curv, "laplacian_energy": raw,
                     "kernel_states": states})
    return pd.DataFrame(rows).sort_values("anova_energy", ascending=False)


# ---------------------------------------------------------------------------
# NEW: main-effect FUNCTION plots -- f_j(x) vs x, not just the scalar
# importance/curvature summary in main_effect_table(). No cyclic features
# in this dataset, so (unlike the bike-share version) there's no
# reconstruct_cyclic_value() step -- every feature's x-axis is just its
# own raw value.
# ---------------------------------------------------------------------------

def compute_main_effect_curve(full, Xeval, background, changed_cols):
    """Same first-order fANOVA contrast computation as inside
    main_effect_table(), factored out standalone so it can be called again
    here to get the per-point curve (not just the summary row) without
    modifying main_effect_table() itself."""
    base = full.predict(background)[None, :]
    h = background_predictions(full, Xeval, background, changed_cols)
    psi = h - base
    comp = psi.mean(axis=1)
    comp = comp - comp.mean()
    return comp


def plot_main_effect_functions(full, B, Xte_raw, raw_names, changed_of,
                                background, out_dir, n_bins=30):
    """For every main effect, save two plots of the fANOVA main-effect
    FUNCTION f_j(x) against the feature's own value x:
      - "_raw.png": every test-set point plotted as-is (unsmoothed scatter)
      - "_binned.png": the same points binned into n_bins equal-width bins
        along x, with the mean f_j value per bin plotted as a line -- a
        cleaner view of the underlying shape, at the cost of within-bin
        detail.

    x-axis values: the feature's own column in Xte_raw (the original,
    pre-generic-rename test set) -- e.g. MedInc, HouseAge, Latitude, used
    directly. No reconstruction step needed since nothing here is
    cyclic-encoded.

    y-axis values: the SAME per-point main-effect contribution
    main_effect_table() summarizes into anova_energy/model_kernel_curvature
    for that feature, recomputed here via compute_main_effect_curve() so
    this function can plot the full per-point curve.

    B and Xte_raw must be row-aligned (same row order) -- true here because
    B is built as Xte_raw.copy() with only the column names changed,
    earlier in main().
    """
    print("\nplotting main-effect functions (f_j(x) vs x) for each feature")
    for name in raw_names:
        comp = compute_main_effect_curve(full, B, background, changed_of[name])
        x_vals = Xte_raw[name].to_numpy()
        xlabel = name
        ylabel = f"fANOVA main effect  f_{{{name}}}(x)  (centered)"

        # --- raw (unsmoothed) scatter ---
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(x_vals, comp, s=6, alpha=0.25, color="#4C72B0")
        ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"Main effect (raw): {name}")
        fig.tight_layout()
        raw_path = out_dir / f"california_housing_main_effect_{name}_raw.png"
        fig.savefig(raw_path, dpi=150)
        plt.close(fig)

        # --- binned mean curve ---
        lo, hi = float(np.min(x_vals)), float(np.max(x_vals))
        bin_edges = np.linspace(lo, hi, n_bins + 1)
        # np.digitize with these interior edges maps values to bins
        # 0..n_bins-1; clip handles the max value landing exactly on the
        # rightmost edge.
        bin_idx = np.clip(np.digitize(x_vals, bin_edges[1:-1]), 0, n_bins - 1)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        bin_means = np.array([
            comp[bin_idx == b].mean() if np.any(bin_idx == b) else np.nan
            for b in range(n_bins)
        ])

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(bin_centers, bin_means, marker="o", color="#DD8452")
        ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel + f"  ({n_bins}-bin mean)")
        ax.set_title(f"Main effect (binned): {name}")
        fig.tight_layout()
        binned_path = out_dir / f"california_housing_main_effect_{name}_binned.png"
        fig.savefig(binned_path, dpi=150)
        plt.close(fig)

        print(f"  {name}: saved {raw_path.name}, {binned_path.name}")


def load_and_prepare():
    """Fetch California Housing (via sklearn) and return (X, y, raw_names)."""
    print("loading California Housing data (sklearn.datasets.fetch_california_housing)")
    X, y_raw = C.load_california_housing()

    # No-op today (C.DROP is empty), kept for structural symmetry with the
    # bike-share script in case a column ever needs excluding later.
    X = X.drop(columns=[c for c in C.DROP if c in X.columns])

    y = np.log1p(y_raw) if C.LOG1P_TARGET else y_raw

    # No-op today (C.CAT_FEATURES is empty): every column here is already
    # a continuous numeric quantity.
    for c in C.CAT_FEATURES:
        if c in X.columns:
            X[c] = X[c].astype("category")

    bad = [c for c in X.columns if X[c].dtype == object]
    if bad:
        raise TypeError(
            f"Non-numeric, non-categorical columns remain after preprocessing: {bad}. "
            "Add them to config.DROP or config.CAT_FEATURES."
        )

    return X, y, list(X.columns)


def main():
    X, y, raw_names = load_and_prepare()
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=C.TEST_SIZE,
                                           random_state=C.RANDOM_STATE)
    # Reset indices so later .iloc-based positional access (in
    # background_predictions) lines up cleanly.
    Xtr, Xte = Xtr.reset_index(drop=True), Xte.reset_index(drop=True)
    # Rename columns to generic f0, f1, ... -- keeps XGBoost's
    # interaction_constraints syntax simple/stable, independent of the
    # original column names.
    cols = [f"f{i}" for i in range(X.shape[1])]
    raw_to_col = dict(zip(raw_names, cols))   # 1:1 -- no cyclic split, so no grouping needed
    A, B = Xtr.copy(), Xte.copy(); A.columns = cols; B.columns = cols

    # Load the top-8 candidate interaction pairs from the prior screening
    # step (see california_housing_oof_depth2_screen.py).
    screened_path = C.OUT_DIR / "california_housing_screened_pairs.csv"
    if not screened_path.exists():
        raise FileNotFoundError(
            f"Expected screened-pairs file not found: {screened_path}\n"
            "Run the depth-2 OOF interaction screen on the California "
            "Housing data first and save its output (feature_1, feature_2, "
            "screen_gain columns) to this path before running this script."
        )
    screened = pd.read_csv(screened_path).head(8)
    # Guard against a stale screened-pairs CSV referencing a feature name
    # that no longer exists, with a specific error instead of a bare
    # KeyError buried in a comprehension.
    unknown = sorted({
        name for row in screened.itertuples(index=False)
        for name in (row.feature_1, row.feature_2)
        if name not in raw_to_col
    })
    if unknown:
        raise KeyError(
            f"{screened_path.name} references feature(s) not in the current "
            f"feature set: {unknown}. Re-run "
            "california_housing_oof_depth2_screen.py to regenerate it, "
            "then re-run this script."
        )
    pairs = [(raw_to_col[row.feature_1], raw_to_col[row.feature_2])
             for row in screened.itertuples(index=False)]
    target_desc = f"log1p({C.TARGET})" if C.LOG1P_TARGET else C.TARGET
    print(f"fit final deep residual model; target={target_desc}")
    # Stage 1 (GAM): depth-10 XGBoost, but interaction_constraints=[[c] for
    # c in cols] forces every split path to stay within a single feature --
    # despite depth 10, it cannot build cross-feature interactions. Result
    # is effectively additive per-feature, with deep single-feature
    # nonlinearity allowed.
    gam = xgb(10, [[c] for c in cols]).fit(A, ytr)
    # Stage 2 (deep residual): depth-6 XGBoost fit on the GAM's residuals
    # (ytr - gam.predict(A)), constrained so only the 8 screened pairs may
    # interact. This is "the depth-6 residual model constrained by those
    # pairs" from the module docstring.
    deep = xgb(6, [list(pair) for pair in pairs]).fit(A, ytr - gam.predict(A))
    pred = gam.predict(B) + deep.predict(B)
    print(f"  held-out {target_desc} MAE: GAM+screened-deep={np.abs(pred-yte).mean():.4f}")

    # Fixed-seed sample of 96 background rows from the training set, used
    # as the reference distribution for all fANOVA interventions below.
    rng = np.random.default_rng(C.RANDOM_STATE)
    background = A.iloc[rng.choice(len(A), size=96, replace=False)].reset_index(drop=True)
    # Deep model's baseline predictions on the background set alone (no
    # eval-point values swapped in).
    base = deep.predict(background)[None, :]
    records = []
    print("construct deep fANOVA PSD kernels and component curvature")
    for (a, b), row in zip(pairs, screened.itertuples(index=False)):
        # Classical fANOVA inclusion-exclusion for a second-order (pairwise)
        # interaction, evaluated on the test set B against the background
        # sample, using the deep residual model only:
        #
        #   f_{jk}(x) ~= E_b[model(x_j, x_k, b_rest)]
        #              - E_b[model(x_j, b_rest)]
        #              - E_b[model(x_k, b_rest)]
        #              + E_b[model(b)]
        #
        hab = background_predictions(deep, B, background, [a, b])  # both features swapped
        ha = background_predictions(deep, B, background, [a])      # only a swapped
        hb = background_predictions(deep, B, background, [b])      # only b swapped
        # Inclusion-exclusion combination: cancels both main effects and the
        # baseline, isolating the pure pairwise interaction contrast. This
        # psi matrix (per-eval-point, per-background-draw) is exactly what
        # forms K_S = Psi_S Psi_S' / M -- the PSD kernel induced by the
        # deep model's actual interaction structure, per the module
        # docstring.
        psi = hab - ha - hb + base                 # finite fANOVA feature map
        component = psi.mean(axis=1)               # f_{jk} under this measure
        component = component - component.mean()    # center: A_S is variance, not 2nd moment
        # Curvature diagnostic for this interaction: how smooth/coherent
        # the interaction component is over its own model-induced graph.
        curvature, raw, states = model_kernel_curvature(psi / np.sqrt(psi.shape[1]), component)
        records.append({"feature_1": row.feature_1, "feature_2": row.feature_2,
                        "screen_gain": row.screen_gain,
                        "anova_energy": float(np.mean(component ** 2)),
                        "component_std": float(np.std(component)),
                        "model_kernel_curvature": curvature,
                        "laplacian_energy": raw, "kernel_states": states,
                        "kernel_rank": int(np.linalg.matrix_rank(psi))})
    out = pd.DataFrame(records).sort_values("anova_energy", ascending=False)
    print("\nDeep-model pair fANOVA importance + model-kernel curvature:")
    print(out.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    out.to_csv(C.OUT_DIR / "california_housing_deep_anova_kernel_curvature.csv", index=False)
    np.savez(C.OUT_DIR / "california_housing_deep_anova_kernel_curvature.npz",
             **{col: out[col].to_numpy() for col in out.columns})
    print(f"\nsaved -> {C.OUT_DIR / 'california_housing_deep_anova_kernel_curvature.csv'}")

    # Wrap GAM + deep residual as a single combined model so main effects
    # are computed against the FULL model, not just the residual stage.
    full = SumModel(gam, deep)
    changed_of = {name: [raw_to_col[name]] for name in raw_names}
    mains = main_effect_table(full, B, background, raw_names, changed_of)
    print("\nMain-effect fANOVA importance + model-kernel curvature:")
    print(mains.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    mains.to_csv(C.OUT_DIR / "california_housing_main_effect_curvature.csv", index=False)

    # NEW: plot each main effect as a function of its own feature value
    # (both a raw scatter and a binned-mean curve per feature), saved to
    # C.OUT_DIR. Xte (not B) is passed for the x-axis values because it
    # still has the original, readable column names (MedInc, HouseAge,
    # Latitude, ...) -- B has already been renamed to generic f0/f1/...
    # columns for modeling.
    plot_main_effect_functions(full, B, Xte, raw_names, changed_of,
                                background, C.OUT_DIR)


if __name__ == "__main__":
    main()

loading California Housing data (sklearn.datasets.fetch_california_housing)
fit final deep residual model; target=log1p(MedHouseVal)
  held-out log1p(MedHouseVal) MAE: GAM+screened-deep=0.1080
construct deep fANOVA PSD kernels and component curvature

Deep-model pair fANOVA importance + model-kernel curvature:
feature_1 feature_2  screen_gain  anova_energy  component_std  model_kernel_curvature  laplacian_energy  kernel_states  kernel_rank
 Latitude Longitude      0.01309       0.00271        0.05206                 0.01756           0.00005           2798           96
 AveRooms  AveOccup      0.00061       0.00076        0.02751                 0.06319           0.00005           3967           96
   MedInc Longitude      0.00083       0.00046        0.02142                 0.03783           0.00002           3920           96
   MedInc  AveOccup      0.00070       0.00035        0.01859                 0.11353           0.00004           3984           96
   MedInc  Latitude      0.0

In [4]:
"""Deep-model fANOVA interaction kernels and curvature for the synthetic
curvature/importance validation dataset.

Stage 1: XGBoost GAM (additive, all 8 features isolated).
Stage 2: SKIPPED. Unlike the bike-share and California-housing studies,
the 4 candidate interaction pairs here are not discovered by an OOF
depth-2 screen -- they're taken directly from config.GROUND_TRUTH, since
this dataset was built BY CONSTRUCTION to have exactly these 4 pairs (and
only these 4) carry real interaction signal. (If you want to check that
the screen would have found them on its own, see
synthetic_curvature_oof_depth2_screen.py -- it's independent of this
script and not required to run it.)
Stage 3: a depth-6 residual model constrained to those 4 known pairs.

For each pair S={j,k}, Psi_S(x) is a finite fANOVA contrast feature map
over empirical background draws. K_S = Psi_S Psi_S' / M is therefore PSD
and is induced by the FINAL deep model, not by raw feature distance.

# ---------------------------------------------------------------------------
# WHAT THIS SCRIPT IS ACTUALLY TESTING
# config.generate_synthetic_data() builds y from 4 independent components,
# each a genuine (non-additively-separable) interaction between one pair
# of features, with curvature (shape narrowness) and variance (amplitude)
# controlled independently:
#
#     pair        pattern        design curvature   design variance
#     (f1, f2)    small bump     high               low
#     (f3, f4)    smooth shift   low                high
#     (f5, f6)    tall peak      high               high (most important)
#     (f7, f8)    blip           low                low (least important)
#
# This script runs those 4 pairs through the SAME machinery as the other
# two studies (model_kernel_curvature's Q_S, anova_energy) and prints a
# direct comparison against the design intent above, so you can see
# whether the pipeline's curvature/importance measures actually recover
# this known structure -- rather than just asserting they should.
#
# NAMING / FILE SEPARATION: own config module
# (synthetic_curvature_config.py), own C.OUT_DIR
# (Downloads/synthetic_curvature_artifacts), own uniquely-named outputs
# (synthetic_curvature_*). Nothing here touches the bike-share or
# California-housing configs, folders, or files.
# ---------------------------------------------------------------------------
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBRegressor

_DOWNLOADS = Path.home() / "Downloads"
if str(_DOWNLOADS) not in sys.path:
    sys.path.insert(0, str(_DOWNLOADS))

import synthetic_curvature_config as C


def xgb(depth, constraints=None):
    kw = dict(n_estimators=C.SWEEP_ITERS, max_depth=depth, learning_rate=C.SWEEP_LR,
              objective="reg:squarederror", tree_method="hist", n_jobs=-1,
              random_state=C.RANDOM_STATE)
    if constraints is not None:
        kw["interaction_constraints"] = constraints
    return XGBRegressor(**kw)


def background_predictions(model, Xeval, background, changed, chunk=48):
    """Matrix [point, background]: model(x_changed, b_unchanged)."""
    n, m = len(Xeval), len(background)
    ans = np.empty((n, m))
    for lo in range(0, n, chunk):
        hi = min(lo + chunk, n)
        frame = pd.concat([background] * (hi - lo), ignore_index=True)
        for col in changed:
            frame[col] = Xeval.iloc[lo:hi][col].repeat(m).reset_index(drop=True)
        ans[lo:hi] = model.predict(frame).reshape(hi - lo, m)
    return ans


def model_kernel_curvature(psi, component, k=C.KNN, n_perm=200, seed=0):
    """Support-invariant component curvature on K=Psi Psi'/M's feature graph.

    The roughness R(g) is the count-weighted mean squared graph Laplacian of the
    component. We normalize it by its mean under random permutations of the
    component values over the graph nodes, R(g o pi). The permutation null absorbs
    the graph's size and structure, so Q_S is comparable across components with
    very different numbers of distinct states -- a binary feature on a two-node
    graph no longer inflates. It is also amplitude-invariant, since numerator and
    null scale together. Q_S below 1 is smoother than chance; a degenerate
    few-node graph sits near 1.
    """
    states, inv, counts = np.unique(psi, axis=0, return_inverse=True, return_counts=True)
    values = np.array([component[inv == q].mean() for q in range(len(states))])
    if len(states) < 2 or np.var(component) < 1e-12:
        return 0.0, 0.0, len(states)
    kk = min(k, len(states) - 1)
    d, ix = NearestNeighbors(n_neighbors=kk + 1).fit(states).kneighbors(states)
    d, ix = d[:, 1:], ix[:, 1:]
    fallback = np.median(d[d > 1e-12]) if np.any(d > 1e-12) else 1.0
    scale = np.where(d[:, -1] > 1e-12, d[:, -1], fallback)
    w = np.exp(-d ** 2 / (2 * scale[:, None] ** 2 + 1e-12)); w /= w.sum(axis=1, keepdims=True)

    def roughness(v):
        lap = v - (w * v[ix]).sum(axis=1)
        return float(np.average(lap ** 2, weights=counts))

    raw = roughness(values)
    rng = np.random.default_rng(seed)
    null = float(np.mean([roughness(rng.permutation(values)) for _ in range(n_perm)]))
    return (raw / null if null > 0 else 0.0), raw, len(states)


class SumModel:
    """Full model f = f_1 + g_deep, so first-order contrasts give the model's
    main effects and second-order contrasts give the interactions."""
    def __init__(self, a, b):
        self.a, self.b = a, b

    def predict(self, X):
        return self.a.predict(X) + self.b.predict(X)


def main_effect_table(full, Xeval, background, features, changed_of):
    """Importance A_j and curvature Q_j for each main effect (first-order
    contrast of the full model), computed like the pair components."""
    base = full.predict(background)[None, :]
    rows = []
    for name in features:
        h = background_predictions(full, Xeval, background, changed_of[name])
        psi = h - base                              # first-order fANOVA contrast
        comp = psi.mean(axis=1)
        comp = comp - comp.mean()                   # center: A_j is variance
        curv, raw, states = model_kernel_curvature(psi / np.sqrt(psi.shape[1]), comp)
        rows.append({"feature": name, "anova_energy": float(np.mean(comp ** 2)),
                     "component_std": float(np.std(comp)),
                     "model_kernel_curvature": curv, "laplacian_energy": raw,
                     "kernel_states": states})
    return pd.DataFrame(rows).sort_values("anova_energy", ascending=False)


def load_and_prepare():
    """Generate the synthetic dataset and return (X, y, raw_names)."""
    print(f"generating synthetic data (n={C.N_SAMPLES})")
    X, y = C.generate_synthetic_data(n=C.N_SAMPLES, seed=C.RANDOM_STATE)
    return X, y, list(X.columns)


def main():
    X, y, raw_names = load_and_prepare()
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=C.TEST_SIZE,
                                           random_state=C.RANDOM_STATE)
    Xtr, Xte = Xtr.reset_index(drop=True), Xte.reset_index(drop=True)
    cols = [f"f{i}" for i in range(X.shape[1])]
    raw_to_col = dict(zip(raw_names, cols))
    A, B = Xtr.copy(), Xte.copy(); A.columns = cols; B.columns = cols

    # Pairs come straight from GROUND_TRUTH -- no screened-pairs CSV, no
    # screening step. This is only valid because the dataset was built to
    # have exactly these 4 interactions and nothing else; on real data you
    # would not know this in advance, which is exactly what the OOF screen
    # is for in the other two studies.
    ground_truth_pairs = list(C.GROUND_TRUTH.keys())
    pairs = [(raw_to_col[f1], raw_to_col[f2]) for f1, f2 in ground_truth_pairs]
    target_desc = f"log1p({C.TARGET})" if C.LOG1P_TARGET else C.TARGET
    print(f"fit final deep residual model; target={target_desc}")
    print(f"using {len(pairs)} known ground-truth pairs (screening skipped): "
          f"{ground_truth_pairs}")

    # Stage 1 (GAM): depth-10, all 8 features isolated -- purely additive.
    gam = xgb(10, [[c] for c in cols]).fit(A, ytr)
    # Stage 2: depth-6, constrained to exactly the 4 known pairs.
    deep = xgb(6, [list(pair) for pair in pairs]).fit(A, ytr - gam.predict(A))
    pred = gam.predict(B) + deep.predict(B)
    print(f"  held-out {target_desc} MAE: GAM+known-pairs-deep={np.abs(pred-yte).mean():.4f}")

    rng = np.random.default_rng(C.RANDOM_STATE)
    background = A.iloc[rng.choice(len(A), size=96, replace=False)].reset_index(drop=True)
    base = deep.predict(background)[None, :]
    records = []
    print("construct deep fANOVA PSD kernels and component curvature")
    for (a, b), (f1, f2) in zip(pairs, ground_truth_pairs):
        hab = background_predictions(deep, B, background, [a, b])
        ha = background_predictions(deep, B, background, [a])
        hb = background_predictions(deep, B, background, [b])
        psi = hab - ha - hb + base
        component = psi.mean(axis=1)
        component = component - component.mean()
        curvature, raw, states = model_kernel_curvature(psi / np.sqrt(psi.shape[1]), component)
        truth = C.GROUND_TRUTH[(f1, f2)]
        records.append({"feature_1": f1, "feature_2": f2,
                        "pattern": truth["pattern"],
                        "expected_curvature": truth["expected_curvature"],
                        "expected_importance": truth["expected_importance"],
                        "anova_energy": float(np.mean(component ** 2)),
                        "component_std": float(np.std(component)),
                        "model_kernel_curvature": curvature,
                        "laplacian_energy": raw, "kernel_states": states,
                        "kernel_rank": int(np.linalg.matrix_rank(psi))})
    out = pd.DataFrame(records).sort_values("anova_energy", ascending=False)
    print("\nDesigned-pair fANOVA importance + model-kernel curvature "
          "(measured vs. design intent):")
    print(out.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    out.to_csv(C.OUT_DIR / "synthetic_curvature_pair_results.csv", index=False)
    print(f"\nsaved -> {C.OUT_DIR / 'synthetic_curvature_pair_results.csv'}")

    full = SumModel(gam, deep)
    changed_of = {name: [raw_to_col[name]] for name in raw_names}
    mains = main_effect_table(full, B, background, raw_names, changed_of)
    print("\nMain-effect fANOVA importance + model-kernel curvature "
          "(should be near-zero for all 8 features -- every true signal "
          "here is a pairwise interaction, not a main effect):")
    print(mains.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    mains.to_csv(C.OUT_DIR / "synthetic_curvature_main_effect_curvature.csv", index=False)


if __name__ == "__main__":
    main()

generating synthetic data (n=20000)
fit final deep residual model; target=y
using 4 known ground-truth pairs (screening skipped): [('f1', 'f2'), ('f3', 'f4'), ('f5', 'f6'), ('f7', 'f8')]
  held-out y MAE: GAM+known-pairs-deep=0.1527
construct deep fANOVA PSD kernels and component curvature

Designed-pair fANOVA importance + model-kernel curvature (measured vs. design intent):
feature_1 feature_2      pattern expected_curvature  expected_importance  anova_energy  component_std  model_kernel_curvature  laplacian_energy  kernel_states  kernel_rank
       f5        f6    tall_peak               high              highest       0.19490        0.44148                 0.00175           0.00037           3805           96
       f3        f4 smooth_shift                low                 high       0.01645        0.12825                 0.00650           0.00011           3866           96
       f1        f2   small_bump               high                  low       0.00059        0.02438    

In [6]:
"""Validate that a depth-d XGBoost tree ensemble can represent (and
requires) depth >= k to model a genuine k-way feature interaction.

# ---------------------------------------------------------------------------
# THE TEST
#
# For k independent, mean-zero features, the PRODUCT
#
#     y = x1 * x2 * ... * xk
#
# is a "pure" k-way interaction: every lower-order fANOVA component (every
# main effect, every 2-way, 3-way, ..., (k-1)-way sub-interaction) is
# EXACTLY ZERO in expectation. This follows from symmetry: any subset of
# fewer than k of the x_i's, averaged over the remaining features (which
# are independent and mean-zero), integrates to zero. So there is no
# lower-order signal for a shallow model to partially latch onto -- the
# ONLY way to explain any of the variance in y is to use all k features
# together.
#
# A single decision tree of max_depth = d can use at most d distinct
# features along any one root-to-leaf path, so a single tree with d < k
# cannot represent x1*...*xk on any leaf. Boosting (summing many such
# trees) can, in principle, still approximate SOME of a higher-order
# function by superposing many low-order pieces -- so this isn't a hard
# mathematical impossibility for the ensemble as a whole -- but it should
# be dramatically less sample/parameter-efficient than a single tree that
# is deep enough to just represent the interaction directly.
#
# TWO SEPARATE QUESTIONS, TWO SEPARATE CHECKS
#   - NECESSITY: does depth < k unlock ANY signal at all? Population R^2
#     for depth < k should be EXACTLY 0 (same symmetry argument that
#     built the target), so we check the first depth where OOF R^2 clears
#     a small ABSOLUTE epsilon above 0 -- this should land exactly at
#     depth == k, and any depth < k should sit at ~0 (modulo CV noise).
#   - SUFFICIENCY / RESOLUTION: once depth >= k, how much depth does it
#     take to fit the interaction WELL (not just detect it)? At exactly
#     depth = k, a tree gets at most one split per relevant feature -- a
#     coarse octant-level approximation of a smooth product surface.
#     Resolving it well needs multiple splits per feature, hence more
#     depth than k, especially for larger k (more dimensions to resolve).
#     We check the first depth clearing a substantial ABSOLUTE R^2
#     threshold, and separately report whether each k's curve has
#     actually plateaued by MAX_DEPTH (small R^2 delta over the last few
#     depths) so higher orders aren't judged against a ceiling they
#     haven't reached yet.
#
# We sweep true order k = 1..MAX_ORDER and tree depth d = 1..MAX_DEPTH,
# fit an XGBoost regressor via K-fold out-of-fold cross-validation for
# each (k, d) combination, and report held-out R^2 as a k x d grid.
# ---------------------------------------------------------------------------
"""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

# --- config ---
OUT_DIR = Path.home() / "Downloads" / "depth_interaction_validation_artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SAMPLES = 20_000
N_SPLITS = 5              # K-fold OOF
MAX_ORDER = 5              # true interaction orders tested: k = 1..MAX_ORDER
MAX_DEPTH = 12              # tree depths tested: d = 1..MAX_DEPTH -- pushed
                            # well past MAX_ORDER so higher-order curves
                            # (k=4, k=5) have room to actually plateau
                            # before we judge them, instead of being cut
                            # off mid-climb like at MAX_DEPTH=7
N_ESTIMATORS = 800          # raised alongside MAX_DEPTH so higher-order,
                            # higher-dimensional products have enough
                            # boosting rounds to resolve a fine partition
LEARNING_RATE = 0.1
NOISE_FRAC = 0.05           # noise std as a fraction of each order's own
                            # signal std, so signal-to-noise ratio is
                            # comparable across orders even though
                            # Var[x1*...*xk] shrinks geometrically with k

# Necessity check: OOF R^2 should be ~0 (population truth is EXACTLY 0)
# for every depth < k, and should clear this small absolute epsilon right
# at depth == k.
NEAR_ZERO_EPS = 0.02
# Sufficiency check: substantial absolute R^2 threshold, used to gauge how
# much MORE depth beyond k is needed to fit (not just detect) the
# interaction well.
ABS_THRESHOLD = 0.5
# Plateau check: if OOF R^2 changes by less than this over the last few
# depths tested, treat the curve as having plateaued by MAX_DEPTH.
PLATEAU_TOL = 0.01


def make_pure_kway_data(k: int, n: int, seed: int):
    """y = x1 * x2 * ... * xk (+ noise), x_i iid Uniform(-1, 1).

    Zero mean, independent features -> every fANOVA component of order
    < k vanishes in expectation, so this is a "pure" k-way interaction
    with no lower-order signal at all.
    """
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1.0, 1.0, size=(n, k))
    y_clean = X.prod(axis=1)
    noise_std = NOISE_FRAC * y_clean.std()
    y = y_clean + rng.normal(0.0, noise_std, size=n)
    cols = [f"x{i+1}" for i in range(k)]
    return pd.DataFrame(X, columns=cols), y


def xgb(depth):
    return XGBRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=depth,
        learning_rate=LEARNING_RATE,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def oof_r2(X: pd.DataFrame, y: np.ndarray, depth: int, seed: int) -> float:
    """K-fold out-of-fold R^2 for one (data, depth) combination. OOF
    (rather than train-set) R^2 matters here: a sufficiently deep/many-tree
    model could memorize the training set regardless of whether it's
    actually representing the interaction correctly, which would mask
    exactly the depth-insufficiency effect we're trying to detect."""
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    pred = np.empty(len(X))
    for train_idx, test_idx in kf.split(X):
        model = xgb(depth)
        model.fit(X.iloc[train_idx], y[train_idx])
        pred[test_idx] = model.predict(X.iloc[test_idx])
    return float(1.0 - np.mean((y - pred) ** 2) / np.var(y))


def main():
    records = []
    for k in range(1, MAX_ORDER + 1):
        print(f"\n=== true interaction order k={k} ===")
        X, y = make_pure_kway_data(k, N_SAMPLES, seed=RANDOM_STATE)
        for depth in range(1, MAX_DEPTH + 1):
            r2 = oof_r2(X, y, depth, seed=RANDOM_STATE)
            sufficient = depth >= k
            records.append({"true_order": k, "depth": depth, "oof_r2": r2,
                             "depth_sufficient": sufficient})
            flag = "OK  " if sufficient else "n/a "
            print(f"  depth={depth} [{flag}depth>=k]  OOF R^2 = {r2:.4f}")

    out = pd.DataFrame(records)
    pivot = out.pivot(index="depth", columns="true_order", values="oof_r2")
    print("\nOOF R^2 grid (rows=tree depth, columns=true interaction order k):")
    print(pivot.to_string(float_format=lambda v: f"{v:.4f}"))

    out_path = OUT_DIR / "depth_interaction_validation_results.csv"
    out.to_csv(out_path, index=False)
    print(f"\nsaved -> {out_path}")

    # --- NECESSITY check: first depth clearing a small ABSOLUTE epsilon
    # above zero. Population R^2 for depth < k is exactly 0, so this
    # should land exactly at depth == k, with every depth < k reading
    # ~0 (modulo CV noise) beforehand.
    print(f"\nNecessity check: first depth with OOF R^2 > {NEAR_ZERO_EPS} "
          "(population truth is exactly 0 below depth=k), vs. true order k")
    for k in range(1, MAX_ORDER + 1):
        sub = out[out.true_order == k].sort_values("depth")
        hit = sub[sub.oof_r2 > NEAR_ZERO_EPS]
        first_depth = int(hit["depth"].iloc[0]) if len(hit) else None
        match = "MATCHES k" if first_depth == k else "does NOT match k"
        below_k = sub[sub.depth < k]["oof_r2"]
        below_k_desc = (f"max R^2 for depth<k = {below_k.max():.4f}"
                         if len(below_k) else "(k=1: no depth<k to check)")
        print(f"  k={k}: first depth clearing epsilon = {first_depth}  ({match});  {below_k_desc}")

    # --- SUFFICIENCY / RESOLUTION check: first depth clearing a
    # substantial absolute R^2 threshold -- how much depth BEYOND k does
    # it take to fit the interaction well, not just detect it.
    print(f"\nSufficiency check: first depth with OOF R^2 >= {ABS_THRESHOLD}, vs. true order k")
    for k in range(1, MAX_ORDER + 1):
        sub = out[out.true_order == k].sort_values("depth")
        hit = sub[sub.oof_r2 >= ABS_THRESHOLD]
        first_depth = int(hit["depth"].iloc[0]) if len(hit) else None
        extra = f"(+{first_depth - k} beyond k)" if first_depth is not None else "(not reached by MAX_DEPTH)"
        print(f"  k={k}: first depth reaching R^2>={ABS_THRESHOLD} = {first_depth}  {extra}")

    # --- PLATEAU check: has each k's curve actually leveled off by
    # MAX_DEPTH, or is it still climbing (in which case ABS_THRESHOLD /
    # ceiling comparisons for that k are premature)?
    print(f"\nPlateau check: OOF R^2 change over the last 2 depths tested (tol={PLATEAU_TOL})")
    for k in range(1, MAX_ORDER + 1):
        sub = out[out.true_order == k].sort_values("depth")
        delta = float(sub["oof_r2"].iloc[-1] - sub["oof_r2"].iloc[-2])
        status = "plateaued" if abs(delta) < PLATEAU_TOL else "STILL CLIMBING"
        print(f"  k={k}: R^2(depth={MAX_DEPTH}) - R^2(depth={MAX_DEPTH-1}) = {delta:+.4f}  [{status}]")

    # --- plot: R^2 vs depth, one line per true order k ---
    fig, ax = plt.subplots(figsize=(9, 6.5))
    cmap = plt.get_cmap("viridis")
    for i, k in enumerate(range(1, MAX_ORDER + 1)):
        sub = out[out.true_order == k].sort_values("depth")
        color = cmap(i / max(1, MAX_ORDER - 1))
        ax.plot(sub["depth"], sub["oof_r2"], marker="o", color=color, label=f"k={k}")
        # Vertical marker at the expected threshold depth == k
        ax.axvline(k, color=color, linestyle="--", linewidth=1, alpha=0.5)
    ax.axhline(0.0, color="gray", linestyle=":", linewidth=1)
    ax.axhline(ABS_THRESHOLD, color="black", linestyle=":", linewidth=1, alpha=0.4)
    ax.set_xlabel("tree max_depth")
    ax.set_ylabel("held-out (OOF) R\u00b2")
    ax.set_title("Depth required to capture a pure k-way interaction\n"
                  "(dashed vertical lines mark depth = k)")
    ax.set_xticks(range(1, MAX_DEPTH + 1))
    ax.legend(title="true order k")
    ax.set_ylim(-0.05, 1.05)
    fig.tight_layout()
    plot_path = OUT_DIR / "depth_interaction_validation_r2_vs_depth.png"
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)
    print(f"\nsaved -> {plot_path}")


if __name__ == "__main__":
    main()


=== true interaction order k=1 ===
  depth=1 [OK  depth>=k]  OOF R^2 = 0.9969
  depth=2 [OK  depth>=k]  OOF R^2 = 0.9974
  depth=3 [OK  depth>=k]  OOF R^2 = 0.9974
  depth=4 [OK  depth>=k]  OOF R^2 = 0.9974
  depth=5 [OK  depth>=k]  OOF R^2 = 0.9974
  depth=6 [OK  depth>=k]  OOF R^2 = 0.9974
  depth=7 [OK  depth>=k]  OOF R^2 = 0.9974
  depth=8 [OK  depth>=k]  OOF R^2 = 0.9974
  depth=9 [OK  depth>=k]  OOF R^2 = 0.9974
  depth=10 [OK  depth>=k]  OOF R^2 = 0.9974
  depth=11 [OK  depth>=k]  OOF R^2 = 0.9974
  depth=12 [OK  depth>=k]  OOF R^2 = 0.9974

=== true interaction order k=2 ===
  depth=1 [n/a depth>=k]  OOF R^2 = -0.0048
  depth=2 [OK  depth>=k]  OOF R^2 = 0.9547
  depth=3 [OK  depth>=k]  OOF R^2 = 0.9957
  depth=4 [OK  depth>=k]  OOF R^2 = 0.9969
  depth=5 [OK  depth>=k]  OOF R^2 = 0.9970
  depth=6 [OK  depth>=k]  OOF R^2 = 0.9969
  depth=7 [OK  depth>=k]  OOF R^2 = 0.9968
  depth=8 [OK  depth>=k]  OOF R^2 = 0.9968
  depth=9 [OK  depth>=k]  OOF R^2 = 0.9968
  depth=10 [OK  depth

In [10]:
"""Validate that a depth-d XGBoost tree ensemble can represent (and
requires) depth >= k to model a genuine k-way feature interaction.

# ---------------------------------------------------------------------------
# THE TEST
#
# For k independent, mean-zero features, the PRODUCT
#
#     y = x1 * x2 * ... * xk
#
# is a "pure" k-way interaction: every lower-order fANOVA component (every
# main effect, every 2-way, 3-way, ..., (k-1)-way sub-interaction) is
# EXACTLY ZERO in expectation. This follows from symmetry: any subset of
# fewer than k of the x_i's, averaged over the remaining features (which
# are independent and mean-zero), integrates to zero. So there is no
# lower-order signal for a shallow model to partially latch onto -- the
# ONLY way to explain any of the variance in y is to use all k features
# together.
#
# A single decision tree of max_depth = d can use at most d distinct
# features along any one root-to-leaf path, so a single tree with d < k
# cannot represent x1*...*xk on any leaf. Boosting (summing many such
# trees) can, in principle, still approximate SOME of a higher-order
# function by superposing many low-order pieces -- so this isn't a hard
# mathematical impossibility for the ensemble as a whole -- but it should
# be dramatically less sample/parameter-efficient than a single tree that
# is deep enough to just represent the interaction directly.
#
# TWO SEPARATE QUESTIONS, TWO SEPARATE CHECKS
#   - NECESSITY: does depth < k unlock ANY signal at all? Population R^2
#     for depth < k should be EXACTLY 0 (same symmetry argument that
#     built the target), so we check the first depth where OOF R^2 clears
#     a small ABSOLUTE epsilon above 0 -- this should land exactly at
#     depth == k, and any depth < k should sit at ~0 (modulo CV noise).
#   - SUFFICIENCY / RESOLUTION: once depth >= k, how much depth does it
#     take to fit the interaction WELL (not just detect it)? At exactly
#     depth = k, a tree gets at most one split per relevant feature -- a
#     coarse octant-level approximation of a smooth product surface.
#     Resolving it well needs multiple splits per feature, hence more
#     depth than k, especially for larger k (more dimensions to resolve).
#     We check the first depth clearing a substantial ABSOLUTE R^2
#     threshold, and separately report whether each k's curve has
#     actually plateaued by MAX_DEPTH (small R^2 delta over the last few
#     depths) so higher orders aren't judged against a ceiling they
#     haven't reached yet.
#
# We sweep true order k = 1..MAX_ORDER and tree depth d = 1..MAX_DEPTH,
# fit an XGBoost regressor via K-fold out-of-fold cross-validation for
# each (k, d) combination, and report held-out R^2 as a k x d grid.
# ---------------------------------------------------------------------------
"""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

# --- config ---
OUT_DIR = Path.home() / "Downloads" / "depth_interaction_validation_artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SAMPLES = 20_000
N_SPLITS = 5              # K-fold OOF
MAX_ORDER = 5              # true interaction orders tested: k = 1..MAX_ORDER
MAX_DEPTH = 12              # tree depths tested: d = 1..MAX_DEPTH -- pushed
                            # well past MAX_ORDER so higher-order curves
                            # (k=4, k=5) have room to actually plateau
                            # before we judge them, instead of being cut
                            # off mid-climb like at MAX_DEPTH=7
N_ESTIMATORS = 5000          # raised alongside MAX_DEPTH so higher-order,
                            # higher-dimensional products have enough
                            # boosting rounds to resolve a fine partition
LEARNING_RATE = 0.1

# Necessity check: OOF R^2 should be ~0 (population truth is EXACTLY 0)
# for every depth < k, and should clear this small absolute epsilon right
# at depth == k.
NEAR_ZERO_EPS = 0.02
# Sufficiency check: substantial absolute R^2 threshold, used to gauge how
# much MORE depth beyond k is needed to fit (not just detect) the
# interaction well.
ABS_THRESHOLD = 0.5
# Plateau check: if OOF R^2 changes by less than this over the last few
# depths tested, treat the curve as having plateaued by MAX_DEPTH.
PLATEAU_TOL = 0.01


def make_pure_kway_data(k: int, n: int, seed: int):
    """y = x1 * x2 * ... * xk, x_i iid Uniform(-1, 1). No noise.

    Zero mean, independent features -> every fANOVA component of order
    < k vanishes in expectation, so this is a "pure" k-way interaction
    with no lower-order signal at all. With no noise added, population
    R^2 for depth < k is exactly 0 and R^2 for a sufficiently expressive
    depth >= k model should approach exactly 1 -- there's no noise floor
    capping how well the interaction can eventually be fit, so any
    plateau below 1.0 at high depth reflects a genuine resolution
    limitation of the tree ensemble, not irreducible noise.
    """
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1.0, 1.0, size=(n, k))
    y = X.prod(axis=1)
    cols = [f"x{i+1}" for i in range(k)]
    return pd.DataFrame(X, columns=cols), y


def xgb(depth):
    return XGBRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=depth,
        learning_rate=LEARNING_RATE,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def oof_r2(X: pd.DataFrame, y: np.ndarray, depth: int, seed: int) -> float:
    """K-fold out-of-fold R^2 for one (data, depth) combination. OOF
    (rather than train-set) R^2 matters here: a sufficiently deep/many-tree
    model could memorize the training set regardless of whether it's
    actually representing the interaction correctly, which would mask
    exactly the depth-insufficiency effect we're trying to detect."""
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    pred = np.empty(len(X))
    for train_idx, test_idx in kf.split(X):
        model = xgb(depth)
        model.fit(X.iloc[train_idx], y[train_idx])
        pred[test_idx] = model.predict(X.iloc[test_idx])
    return float(1.0 - np.mean((y - pred) ** 2) / np.var(y))


def main():
    records = []
    for k in range(1, MAX_ORDER + 1):
        print(f"\n=== true interaction order k={k} ===")
        X, y = make_pure_kway_data(k, N_SAMPLES, seed=RANDOM_STATE)
        for depth in range(1, MAX_DEPTH + 1):
            r2 = oof_r2(X, y, depth, seed=RANDOM_STATE)
            sufficient = depth >= k
            records.append({"true_order": k, "depth": depth, "oof_r2": r2,
                             "depth_sufficient": sufficient})
            flag = "OK  " if sufficient else "n/a "
            print(f"  depth={depth} [{flag}depth>=k]  OOF R^2 = {r2:.4f}")

    out = pd.DataFrame(records)
    pivot = out.pivot(index="depth", columns="true_order", values="oof_r2")
    print("\nOOF R^2 grid (rows=tree depth, columns=true interaction order k):")
    print(pivot.to_string(float_format=lambda v: f"{v:.4f}"))

    out_path = OUT_DIR / "depth_interaction_validation_results.csv"
    out.to_csv(out_path, index=False)
    print(f"\nsaved -> {out_path}")

    # --- NECESSITY check: first depth clearing a small ABSOLUTE epsilon
    # above zero. Population R^2 for depth < k is exactly 0, so this
    # should land exactly at depth == k, with every depth < k reading
    # ~0 (modulo CV noise) beforehand.
    print(f"\nNecessity check: first depth with OOF R^2 > {NEAR_ZERO_EPS} "
          "(population truth is exactly 0 below depth=k), vs. true order k")
    for k in range(1, MAX_ORDER + 1):
        sub = out[out.true_order == k].sort_values("depth")
        hit = sub[sub.oof_r2 > NEAR_ZERO_EPS]
        first_depth = int(hit["depth"].iloc[0]) if len(hit) else None
        match = "MATCHES k" if first_depth == k else "does NOT match k"
        below_k = sub[sub.depth < k]["oof_r2"]
        below_k_desc = (f"max R^2 for depth<k = {below_k.max():.4f}"
                         if len(below_k) else "(k=1: no depth<k to check)")
        print(f"  k={k}: first depth clearing epsilon = {first_depth}  ({match});  {below_k_desc}")

    # --- SUFFICIENCY / RESOLUTION check: first depth clearing a
    # substantial absolute R^2 threshold -- how much depth BEYOND k does
    # it take to fit the interaction well, not just detect it.
    print(f"\nSufficiency check: first depth with OOF R^2 >= {ABS_THRESHOLD}, vs. true order k")
    for k in range(1, MAX_ORDER + 1):
        sub = out[out.true_order == k].sort_values("depth")
        hit = sub[sub.oof_r2 >= ABS_THRESHOLD]
        first_depth = int(hit["depth"].iloc[0]) if len(hit) else None
        extra = f"(+{first_depth - k} beyond k)" if first_depth is not None else "(not reached by MAX_DEPTH)"
        print(f"  k={k}: first depth reaching R^2>={ABS_THRESHOLD} = {first_depth}  {extra}")

    # --- PLATEAU check: has each k's curve actually leveled off by
    # MAX_DEPTH, or is it still climbing (in which case ABS_THRESHOLD /
    # ceiling comparisons for that k are premature)?
    print(f"\nPlateau check: OOF R^2 change over the last 2 depths tested (tol={PLATEAU_TOL})")
    for k in range(1, MAX_ORDER + 1):
        sub = out[out.true_order == k].sort_values("depth")
        delta = float(sub["oof_r2"].iloc[-1] - sub["oof_r2"].iloc[-2])
        status = "plateaued" if abs(delta) < PLATEAU_TOL else "STILL CLIMBING"
        print(f"  k={k}: R^2(depth={MAX_DEPTH}) - R^2(depth={MAX_DEPTH-1}) = {delta:+.4f}  [{status}]")

    # --- plot: R^2 vs depth, one line per true order k ---
    fig, ax = plt.subplots(figsize=(9, 6.5))
    cmap = plt.get_cmap("viridis")
    for i, k in enumerate(range(1, MAX_ORDER + 1)):
        sub = out[out.true_order == k].sort_values("depth")
        color = cmap(i / max(1, MAX_ORDER - 1))
        ax.plot(sub["depth"], sub["oof_r2"], marker="o", color=color, label=f"k={k}")
        # Vertical marker at the expected threshold depth == k
        ax.axvline(k, color=color, linestyle="--", linewidth=1, alpha=0.5)
    ax.axhline(0.0, color="gray", linestyle=":", linewidth=1)
    ax.axhline(ABS_THRESHOLD, color="black", linestyle=":", linewidth=1, alpha=0.4)
    ax.set_xlabel("tree max_depth")
    ax.set_ylabel("held-out (OOF) R\u00b2")
    ax.set_title("Depth required to capture a pure k-way interaction\n"
                  "(dashed vertical lines mark depth = k)")
    ax.set_xticks(range(1, MAX_DEPTH + 1))
    ax.legend(title="true order k")
    ax.set_ylim(-0.05, 1.05)
    fig.tight_layout()
    plot_path = OUT_DIR / "depth_interaction_validation_r2_vs_depth.png"
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)
    print(f"\nsaved -> {plot_path}")


if __name__ == "__main__":
    main()


=== true interaction order k=1 ===
  depth=1 [OK  depth>=k]  OOF R^2 = 0.9996
  depth=2 [OK  depth>=k]  OOF R^2 = 1.0000
  depth=3 [OK  depth>=k]  OOF R^2 = 1.0000
  depth=4 [OK  depth>=k]  OOF R^2 = 1.0000
  depth=5 [OK  depth>=k]  OOF R^2 = 1.0000
  depth=6 [OK  depth>=k]  OOF R^2 = 1.0000
  depth=7 [OK  depth>=k]  OOF R^2 = 1.0000
  depth=8 [OK  depth>=k]  OOF R^2 = 1.0000
  depth=9 [OK  depth>=k]  OOF R^2 = 1.0000
  depth=10 [OK  depth>=k]  OOF R^2 = 1.0000
  depth=11 [OK  depth>=k]  OOF R^2 = 1.0000
  depth=12 [OK  depth>=k]  OOF R^2 = 1.0000

=== true interaction order k=2 ===
  depth=1 [n/a depth>=k]  OOF R^2 = -0.0094
  depth=2 [OK  depth>=k]  OOF R^2 = 0.9931
  depth=3 [OK  depth>=k]  OOF R^2 = 0.9995
  depth=4 [OK  depth>=k]  OOF R^2 = 0.9997
  depth=5 [OK  depth>=k]  OOF R^2 = 0.9998
  depth=6 [OK  depth>=k]  OOF R^2 = 0.9998
  depth=7 [OK  depth>=k]  OOF R^2 = 0.9998
  depth=8 [OK  depth>=k]  OOF R^2 = 0.9998
  depth=9 [OK  depth>=k]  OOF R^2 = 0.9998
  depth=10 [OK  depth

In [11]:
"""Follow-up to depth_interaction_validation.py: hold depth=5 and true
order k=5 fixed, and sweep boosting rounds (via CHECKPOINTS) and learning
rate (eta) instead, to test whether depth=5's weak performance on the
5-way interaction is a RESOLUTION problem that more rounds (and/or a
different eta) can slowly fix, rather than a hard representational
ceiling.

# ---------------------------------------------------------------------------
# WHY THIS TEST
#
# At max_depth = k = 5, a single tree's entire split budget goes toward
# just barely covering all 5 features once each (one split per feature,
# no slack to refine any of them) -- so each tree only ever produces a
# coarse 32-orthant (2^5) approximation of a smooth, continuously-varying
# product surface. Boosting can still refine that partition further
# across MANY rounds (each new tree re-splits the same features at new
# thresholds, incrementally building up a finer joint partition). If
# resolution is the bottleneck (rather than depth=5 being fundamentally
# incapable), more rounds at the SAME depth should keep improving OOF R^2.
#
# eta (learning rate) controls how much of each new tree's correction
# actually gets applied. A smaller eta needs more rounds to reach the same
# place but can settle into a finer, less noisy final partition; a larger
# eta converges faster per round but risks overshooting/coarser resolution.
# Sweeping eta alongside checkpoints separates "just needs more rounds"
# from "needs the right eta to use those rounds well."
#
# CHECKPOINTING (efficiency): rather than retraining a fresh model from
# scratch for every (eta, n_estimators) pair -- 5x the fits for 5
# checkpoints -- each fold is trained ONCE per eta at the max checkpoint's
# tree count, and predictions at every smaller checkpoint are read off
# that same trained model via iteration_range, using only the first N
# trees. XGBoost's boosting is additive and sequential (tree i+1 only ever
# corrects tree i's residual, never retroactively changes earlier trees),
# so predict(..., iteration_range=(0, N)) is EXACTLY what you'd get from
# training with n_estimators=N in the first place -- this is a genuine
# speedup, not an approximation.
#
# Same data-generating function as before: y = x1*x2*x3*x4*x5, x_i iid
# Uniform(-1, 1), no noise -- population R^2 ceiling is exactly 1.0, so
# any shortfall reflects genuine resolution limits, not an irreducible
# noise floor.
# ---------------------------------------------------------------------------
"""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

# --- config ---
OUT_DIR = Path.home() / "Downloads" / "depth_interaction_validation_artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SAMPLES = 20_000
N_SPLITS = 5                 # K-fold OOF
DEPTH = 5
TRUE_ORDER = 5                # k=5, matching DEPTH exactly -- the worst-case cell

CHECKPOINTS = [1000, 3000, 5000, 7000, 9000]   # tree counts to evaluate at
MAX_ESTIMATORS = max(CHECKPOINTS)               # each fold trains once, to this count
ETA_LIST = [0.3, 0.1, 0.05, 0.01]               # xgboost's "eta" == learning_rate


def make_pure_kway_data(k: int, n: int, seed: int):
    """y = x1 * x2 * ... * xk, x_i iid Uniform(-1, 1). No noise -- see
    depth_interaction_validation.py for the full rationale."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1.0, 1.0, size=(n, k))
    y = X.prod(axis=1)
    cols = [f"x{i+1}" for i in range(k)]
    return pd.DataFrame(X, columns=cols), y


def xgb(eta):
    return XGBRegressor(
        n_estimators=MAX_ESTIMATORS,
        max_depth=DEPTH,
        learning_rate=eta,           # eta
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def oof_r2_at_checkpoints(X: pd.DataFrame, y: np.ndarray, eta: float, seed: int) -> dict:
    """K-fold OOF R^2 at every checkpoint in CHECKPOINTS, for one eta.
    Trains exactly N_SPLITS models (one per fold, to MAX_ESTIMATORS trees)
    rather than N_SPLITS x len(CHECKPOINTS) -- predictions at smaller
    checkpoints are read off the same trained model via iteration_range."""
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    preds = {c: np.empty(len(X)) for c in CHECKPOINTS}
    for train_idx, test_idx in kf.split(X):
        model = xgb(eta)
        model.fit(X.iloc[train_idx], y[train_idx])
        Xte = X.iloc[test_idx]
        for c in CHECKPOINTS:
            # iteration_range=(0, c): predict using only the first c trees,
            # exactly reproducing what training with n_estimators=c would
            # have given, without retraining.
            preds[c][test_idx] = model.predict(Xte, iteration_range=(0, c))
    return {c: float(1.0 - np.mean((y - preds[c]) ** 2) / np.var(y)) for c in CHECKPOINTS}


def main():
    print(f"depth={DEPTH}, true_order k={TRUE_ORDER} (worst-case: depth exactly equals k)")
    print(f"checkpoints={CHECKPOINTS}, eta values={ETA_LIST}")
    X, y = make_pure_kway_data(TRUE_ORDER, N_SAMPLES, seed=RANDOM_STATE)

    records = []
    for eta in ETA_LIST:
        print(f"\neta={eta}: training {N_SPLITS} folds x {MAX_ESTIMATORS} trees, "
              f"reading off {len(CHECKPOINTS)} checkpoints each ...")
        r2_by_checkpoint = oof_r2_at_checkpoints(X, y, eta, seed=RANDOM_STATE)
        for c in CHECKPOINTS:
            r2 = r2_by_checkpoint[c]
            print(f"  n_estimators={c}: OOF R^2 = {r2:.4f}")
            records.append({"eta": eta, "depth": DEPTH, "true_order": TRUE_ORDER,
                             "n_estimators": c, "oof_r2": r2})

    out = pd.DataFrame(records)
    pivot = out.pivot(index="n_estimators", columns="eta", values="oof_r2")
    print("\nOOF R^2 grid (rows=n_estimators checkpoint, columns=eta):")
    print(pivot.to_string(float_format=lambda v: f"{v:.4f}"))

    out_path = OUT_DIR / "depth5_k5_estimator_sweep_results.csv"
    out.to_csv(out_path, index=False)
    print(f"\nsaved -> {out_path}")

    # Is R^2 still climbing at the last checkpoint, for each eta?
    print(f"\nR^2 change from n_estimators={CHECKPOINTS[-2]} to {CHECKPOINTS[-1]}, per eta:")
    for eta in ETA_LIST:
        sub = out[out.eta == eta].sort_values("n_estimators")
        delta = float(sub["oof_r2"].iloc[-1] - sub["oof_r2"].iloc[-2])
        status = "still climbing" if delta > 0.01 else "leveled off"
        print(f"  eta={eta}: {delta:+.4f}  [{status}]")

    # Best (eta, n_estimators) combination found
    best = out.loc[out["oof_r2"].idxmax()]
    print(f"\nBest result: eta={best.eta}, n_estimators={int(best.n_estimators)}, "
          f"OOF R^2={best.oof_r2:.4f}")

    # --- plot: R^2 vs n_estimators, one line per eta ---
    fig, ax = plt.subplots(figsize=(8, 6.5))
    cmap = plt.get_cmap("viridis")
    for i, eta in enumerate(ETA_LIST):
        sub = out[out.eta == eta].sort_values("n_estimators")
        color = cmap(i / max(1, len(ETA_LIST) - 1))
        ax.plot(sub["n_estimators"], sub["oof_r2"], marker="o", color=color, label=f"eta={eta}")
    ax.axhline(1.0, color="gray", linestyle=":", linewidth=1, label="ceiling (R\u00b2=1, no noise)")
    ax.set_xlabel("n_estimators (boosting rounds, via checkpoint)")
    ax.set_ylabel("held-out (OOF) R\u00b2")
    ax.set_title(f"depth={DEPTH}, true order k={TRUE_ORDER}: rounds x eta\n"
                 "does more rounds (at the right eta) resolve the interaction?")
    ax.set_ylim(-0.05, 1.05)
    ax.legend()
    fig.tight_layout()
    plot_path = OUT_DIR / "depth5_k5_estimator_sweep.png"
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)
    print(f"saved -> {plot_path}")


if __name__ == "__main__":
    main()

depth=5, true_order k=5 (worst-case: depth exactly equals k)
checkpoints=[1000, 3000, 5000, 7000, 9000], eta values=[0.3, 0.1, 0.05, 0.01]

eta=0.3: training 5 folds x 9000 trees, reading off 5 checkpoints each ...
  n_estimators=1000: OOF R^2 = -0.0703
  n_estimators=3000: OOF R^2 = -0.0565
  n_estimators=5000: OOF R^2 = -0.0565
  n_estimators=7000: OOF R^2 = -0.0565
  n_estimators=9000: OOF R^2 = -0.0565

eta=0.1: training 5 folds x 9000 trees, reading off 5 checkpoints each ...
  n_estimators=1000: OOF R^2 = -0.0606
  n_estimators=3000: OOF R^2 = -0.0327
  n_estimators=5000: OOF R^2 = -0.0308
  n_estimators=7000: OOF R^2 = -0.0308
  n_estimators=9000: OOF R^2 = -0.0308

eta=0.05: training 5 folds x 9000 trees, reading off 5 checkpoints each ...
  n_estimators=1000: OOF R^2 = -0.0647
  n_estimators=3000: OOF R^2 = -0.0459
  n_estimators=5000: OOF R^2 = -0.0331
  n_estimators=7000: OOF R^2 = -0.0284
  n_estimators=9000: OOF R^2 = -0.0284

eta=0.01: training 5 folds x 9000 trees, readi

In [15]:
"""Crack open fitted XGBoost trees to directly check WHERE in each tree
the real signal for a pure k=5 interaction shows up, for depth in {5,6,7}.

# ---------------------------------------------------------------------------
# THE DIAGNOSTIC
#
# Same data as before: y = x1*x2*x3*x4*x5, x_i iid Uniform(-1,1), no noise,
# TRUE_ORDER=5, and this dataset has EXACTLY 5 features (no irrelevant
# filler columns), so every single split necessarily uses one of the 5
# relevant features.
#
# The hypothesis from the earlier discussion: a greedy splitter can't see
# any real signal from splitting on fewer than all 5 features (population
# gain is EXACTLY 0 for any subset of < 5, by the same symmetry argument
# that built the target), so the first several splits down any root-to-
# leaf path are working blind -- only once a path has used ALL 5 distinct
# features does the interaction actually become visible to the greedy
# search.
#
# For every split node in the fitted ensemble, we compute:
#   - "depth": how many splits deep it is (root = 0)
#   - "coverage": the number of DISTINCT features used by that node's
#     path so far (its ancestors' features, plus its own) -- ranges 1..5
#
# coverage is the sharper test: at depth=5 exactly, coverage==5 can only
# happen at the very last split of a path (all 5 splits used a different
# feature). At depth 6 or 7, a path COULD reach coverage==5 earlier (if
# it happened to pick 5 distinct features in its first 5 splits) and then
# either revisit a feature or keep refining -- so depth alone conflates
# "the discovery split" with "extra refinement after discovery" for those
# deeper trees. Bucketing by coverage instead isolates the actual claim:
# gain should be ~0 for coverage < 5, and jump hard at coverage == 5,
# regardless of which depth that happens to land on.
#
# We fit one model per depth in {5, 6, 7}, extract every split's Feature/
# Gain via model.get_booster().trees_to_dataframe(), reconstruct each
# node's ancestor path via the Yes/No child pointers (trees_to_dataframe()
# doesn't give you depth or ancestor info directly -- only parent->child
# links, one row per node), and aggregate Gain by depth and by coverage.
#
# ===========================================================================
# ANNOTATED VERSION -- commentary added throughout below. No logic changed
# from the version this was generated from (including N_ESTIMATORS=5000,
# a deliberate bump from the original 300 so the per-coverage-bucket gain
# aggregates below are averaged over many more split nodes, and the
# coverage==5 signal has more chances to actually get discovered and
# reused across the ensemble at each depth).
# ===========================================================================
"""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")   # headless/non-interactive backend -- this script
                         # only ever saves PNGs, never opens a window, and
                         # Agg avoids needing a display/GUI backend at all
                         # (important when running from a plain terminal).
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# --- config ---
# Reuses the SAME output folder as depth_interaction_validation.py and
# depth5_k5_estimator_sweep.py -- all three scripts are different angles
# on the same underlying question (does depth >= k let XGBoost model a
# pure k-way interaction), so it's convenient to have their outputs land
# side by side.
OUT_DIR = Path.home() / "Downloads" / "depth_interaction_validation_artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SAMPLES = 20_000
TEST_SIZE = 0.20
TRUE_ORDER = 5              # y = x1*...*x5, no noise -- see depth_interaction_validation.py
DEPTHS = [5, 6, 7]          # the three depths under the microscope: exactly
                            # at k, and one/two past it
N_ESTIMATORS = 5000         # bumped up from the original 300 specifically
                            # for this diagnostic -- more trees means more
                            # split nodes at every (depth, coverage)
                            # combination, so the gain aggregates below are
                            # less noisy per bucket, especially for the
                            # rarer high-coverage buckets.
LEARNING_RATE = 0.1


def make_pure_kway_data(k: int, n: int, seed: int):
    """y = x1 * x2 * ... * xk, x_i iid Uniform(-1, 1). No noise."""
    # Deterministic RNG (seeded) so results are exactly reproducible run to
    # run. Uniform(-1,1) features are symmetric and mean-zero, which is
    # what makes every fANOVA component below order k vanish in
    # expectation -- see depth_interaction_validation.py for the full
    # argument.
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1.0, 1.0, size=(n, k))
    # Elementwise product across columns (axis=1) -- the pure k-way
    # interaction target. No additive noise term here at all.
    y = X.prod(axis=1)
    cols = [f"x{i+1}" for i in range(k)]
    return pd.DataFrame(X, columns=cols), y


def fit_model(X_train, y_train, depth):
    """One XGBoost fit at a given max_depth, otherwise identical
    hyperparameters across all three depths tested -- so any difference in
    the resulting gain-by-coverage pattern is attributable to depth alone."""
    model = XGBRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=depth,
        learning_rate=LEARNING_RATE,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    return model


def compute_depth_and_coverage(trees_df: pd.DataFrame) -> pd.DataFrame:
    """Walk each tree from its root (Node 0) via the Yes/No child pointers,
    computing for every SPLIT node:
      - depth: number of splits from the root (root itself = depth 0)
      - coverage: number of distinct features used by this node's path,
        INCLUDING this node's own split feature

    trees_to_dataframe() gives one row per node (split or leaf) with
    parent->child links (Yes/No columns hold child IDs, or NaN for
    leaves) but no depth/ancestor info directly -- that's what this
    reconstructs via an explicit stack-based traversal per tree.
    """
    # Index by the node's own "ID" string (format "{Tree}-{Node}") so
    # indexed.loc[node_id] is an O(1)-ish row lookup during the traversal
    # below, rather than repeatedly filtering the whole DataFrame.
    indexed = trees_df.set_index("ID")
    depth_map: dict = {}       # node_id -> depth (every node, split or leaf)
    coverage_map: dict = {}    # node_id -> coverage (split nodes only;
                                # leaves have no "coverage" of their own)

    # groupby("Tree") to process one tree at a time -- each tree's node
    # IDs and structure are independent of every other tree's.
    for tree_id, _ in trees_df.groupby("Tree"):
        # XGBoost's convention: the root of every tree is always Node 0,
        # so its full ID is "{tree_id}-0".
        root_id = f"{tree_id}-0"
        # Explicit stack-based DFS (rather than recursion) -- avoids
        # Python's recursion depth limit for any unexpectedly deep/large
        # tree, and is just as simple to reason about here.
        #
        # stack entries: (node_id, depth, frozenset of ancestor features
        # NOT including this node's own feature). frozenset (not a plain
        # set) so it's hashable/immutable -- each child gets its OWN copy
        # of the parent's accumulated feature set via union (`|`), so
        # sibling branches never see each other's features by accident.
        stack = [(root_id, 0, frozenset())]
        while stack:
            node_id, d, ancestor_feats = stack.pop()
            depth_map[node_id] = d
            row = indexed.loc[node_id]
            if row["Feature"] == "Leaf":
                # Leaves carry a predicted VALUE in the "Gain" column in
                # trees_to_dataframe() (not an actual split gain), and
                # have no children to recurse into -- nothing further to
                # do for this branch.
                continue
            this_feat = row["Feature"]
            # coverage = how many distinct features this split's path has
            # used by the time you've made THIS split (ancestors' features
            # union this node's own feature).
            coverage_map[node_id] = len(ancestor_feats | {this_feat})
            new_ancestors = ancestor_feats | {this_feat}
            # Push both children (if they exist) with depth+1 and the
            # updated ancestor-feature set. pd.notna() guards against a
            # node that's actually a leaf-like terminal with no Yes/No
            # values (shouldn't normally happen for a non-Leaf row, but
            # cheap to guard against a malformed/edge-case row).
            for child_col in ("Yes", "No"):
                child_id = row[child_col]
                if pd.notna(child_id):
                    stack.append((child_id, d + 1, new_ancestors))

    # Map the two dicts back onto the original DataFrame by ID. Leaves get
    # NaN in "coverage" (never added to coverage_map above), which is
    # correct -- coverage is only a meaningful concept for split nodes.
    trees_df = trees_df.copy()
    trees_df["depth"] = trees_df["ID"].map(depth_map)
    trees_df["coverage"] = trees_df["ID"].map(coverage_map)  # NaN for leaves
    return trees_df


def summarize_gain(trees_df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """Total/mean gain and node count, grouped by depth or coverage,
    restricted to actual split nodes (leaves carry no split gain)."""
    # Filter out leaf rows first -- their "Gain" column holds a leaf
    # weight/value, not a split-quality gain, so including them here would
    # silently corrupt the aggregation.
    splits = trees_df[trees_df["Feature"] != "Leaf"].copy()
    total_gain = splits["Gain"].sum()
    # count: how many split nodes fall in this depth/coverage bucket
    #   (buckets with more nodes are more statistically reliable).
    # sum:   total gain contributed by this bucket across the WHOLE
    #   ensemble (this is what "pct_of_total_gain" is based on).
    # mean:  average gain PER split node in this bucket -- the number to
    #   look at for "is a single split at this depth/coverage typically
    #   worthwhile", independent of how many such splits exist.
    g = splits.groupby(group_col)["Gain"].agg(["count", "sum", "mean"])
    g["pct_of_total_gain"] = 100.0 * g["sum"] / total_gain
    return g.reset_index().rename(columns={group_col: group_col, "sum": "gain_sum", "mean": "gain_mean"})


def main():
    # ONE shared dataset across all three depths -- so depth=5, 6, 7 are
    # compared on the identical train/test split, not independently
    # resampled data that could introduce its own noise into the
    # comparison.
    X, y = make_pure_kway_data(TRUE_ORDER, N_SAMPLES, seed=RANDOM_STATE)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    for depth in DEPTHS:
        print(f"\n{'='*70}\ndepth={depth}, true_order k={TRUE_ORDER}\n{'='*70}")
        model = fit_model(Xtr, ytr, depth)
        # Quick held-out R^2 for context alongside the gain breakdown --
        # not the focus of this script (that's depth_interaction_validation.py
        # and depth5_k5_estimator_sweep.py), but useful to see whether a
        # given depth's gain pattern corresponds to a model that's actually
        # fitting well or still struggling overall.
        pred = model.predict(Xte)
        r2 = 1.0 - np.mean((yte - pred) ** 2) / np.var(yte)
        print(f"held-out R^2 for this fit: {r2:.4f}")

        # trees_to_dataframe(): XGBoost's built-in dump of every node in
        # every tree in the ensemble as a flat table -- this is the "crack
        # open the trees" step. One row per node; split rows have a
        # Feature/Split/Gain, leaf rows have Feature=="Leaf".
        trees_df = model.get_booster().trees_to_dataframe()
        # Reconstruct depth and coverage for every node (see function
        # docstring above) -- this is the part trees_to_dataframe() alone
        # doesn't give you.
        trees_df = compute_depth_and_coverage(trees_df)
        n_split_nodes = int((trees_df["Feature"] != "Leaf").sum())
        print(f"total split nodes across {N_ESTIMATORS} trees: {n_split_nodes}")

        # --- gain by raw depth-within-tree ---
        by_depth = summarize_gain(trees_df, "depth")
        print(f"\nGain by split depth-within-tree (0 = root, i.e. the 1st split on a path):")
        print(by_depth.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
        by_depth.to_csv(OUT_DIR / f"split_gain_by_depth_d{depth}.csv", index =False)

        # --- gain by coverage (distinct features used so far, including this split) ---
        by_cov = summarize_gain(trees_df, "coverage")
        print(f"\nGain by coverage (# distinct features used by this node's path so far, "
              f"including this split) -- expect near-zero until coverage=5:")
        print(by_cov.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
        by_cov.to_csv(OUT_DIR / f"split_gain_by_coverage_d{depth}.csv", index=False)

        # Direct check: mean gain at coverage<5 vs coverage==5. This is
        # the headline number for the whole diagnostic -- a large ratio
        # here is the concrete, quantitative version of "splits before
        # full coverage are blind; the split that completes coverage is
        # where the real signal is."
        below5 = by_cov[by_cov["coverage"] < 5]
        at5 = by_cov[by_cov["coverage"] == 5]
        # Pooled mean across all coverage<5 buckets: total gain from those
        # buckets divided by total split-node count in those buckets (NOT
        # a naive average of the per-bucket means, which would incorrectly
        # weight small buckets as heavily as large ones).
        mean_below5 = float((below5["gain_sum"].sum()) / max(1, below5["count"].sum()))
        mean_at5 = float(at5["gain_mean"].iloc[0]) if len(at5) else float("nan")
        ratio = mean_at5 / mean_below5 if mean_below5 > 0 else float("inf")
        print(f"\nmean gain, coverage<5 (pooled): {mean_below5:.6f}")
        print(f"mean gain, coverage==5:          {mean_at5:.6f}")
        print(f"ratio (coverage==5 / coverage<5): {ratio:.1f}x")

        # --- plot: mean gain by coverage, this depth ---
        fig, ax = plt.subplots(figsize=(6.5, 5))
        # x-axis as strings (not the raw integers) so matplotlib treats
        # coverage as discrete categories with even spacing, rather than
        # a numeric axis that might auto-scale or add unwanted tick
        # spacing logic for what's really a small set of distinct values.
        ax.bar(by_cov["coverage"].astype(int).astype(str), by_cov["gain_mean"], color="#4C72B0")
        ax.set_xlabel("coverage (# distinct features used by this split's path so far)")
        ax.set_ylabel("mean split gain")
        ax.set_title(f"depth={depth}: mean gain by coverage\n(pure k={TRUE_ORDER} interaction, no noise)")
        fig.tight_layout()
        plot_path = OUT_DIR / f"split_gain_by_coverage_d{depth}.png"
        fig.savefig(plot_path, dpi=150)
        plt.close(fig)   # free the figure's memory before the next depth's
                          # iteration creates a new one
        print(f"saved -> {plot_path}")

    print(f"\nall CSVs/plots saved under {OUT_DIR}")


if __name__ == "__main__":
    main()


depth=5, true_order k=5
held-out R^2 for this fit: -0.0110
total split nodes across 5000 trees: 92020

Gain by split depth-within-tree (0 = root, i.e. the 1st split on a path):
 depth  count   gain_sum  gain_mean  pct_of_total_gain
     0   3155   0.955933   0.000303           0.290601
     1   6310  10.511912   0.001666           3.195589
     2  12567  42.831183   0.003408          13.020549
     3  24569  98.318138   0.004002          29.888414
     4  45419 176.333510   0.003882          53.604848

Gain by coverage (# distinct features used by this node's path so far, including this split) -- expect near-zero until coverage=5:
 coverage  count  gain_sum  gain_mean  pct_of_total_gain
  1.00000   7368   5.38613    0.00073            1.63737
  2.00000  29462  72.53579    0.00246           22.05066
  3.00000  41564 161.16164    0.00388           48.99265
  4.00000  12484  69.90046    0.00560           21.24953
  5.00000   1142  19.96665    0.01748            6.06980

mean gain, covera